# Modul 8 — Struktur Penyimpanan Data: SQL vs NoSQL

**Praktikum Data Wrangling (SD25-20007) · Program Studi Sains Data ITERA**

Workbench menyimpan notebook ini di laptop sebagai **`dw-module-08/08_penyimpanan.ipynb`**;
seluruh keluaran modul ditulis ke folder `dw-module-08/` itu. Modul ini membaca keluaran acuan
Modul 4 dan 7 yang disediakan di **`data/acuan/`** (`transaksi_m04`, `produk_m04`, `pelanggan_m04`,
`duplikat_transaksi_m04.csv`, dan `toko_m07.parquet`; bukan keluaran Anda sendiri, sehingga
kesalahan di modul sebelumnya tidak terbawa ke sini), serta `ulasan.jsonl` mentah. Unduh `data-acuan-modul-08.zip` di halaman modul dan ekstrak di folder `data-wrangling/`; isinya masuk ke `data/acuan/`. Dataset mentah disiapkan lewat panel *Dataset di laptop* halaman
modul (Local Runner menyala). Buku ajar: Bab 9, 10.2.4, 10.4, dan 11.4.

| Bagian | Isi notebook ini | Dinilai? |
|---|---|---|
| A–D Prosedur terbimbing | **kode lengkap dan pembahasan** | dijalankan dan lulus checkpoint |
| Latihan 8.1–8.2 | kerangka `TODO` | **ya** — *Latihan Mandiri di Lab* (20% nilai modul) |
| Tugas Individu | kerangka `TODO` | **ya** — *Tugas Individu* (80% nilai modul) |

Bagian A–D bukan untuk disalin tanpa dibaca. Latihan dan tugas memakai **cara berpikir yang
sama** pada rancangan lain (pesanan sebagai dokumen, migrasi skema, dua penyimpanan yang harus
tetap sepakat), dan pertanyaan analisis menanyakan alasan di balik setiap langkah A–D.

**Aturan main**
1. `data/raw/` dan `data/acuan/` bersifat *read-only* — jangan menulis ke sana.
2. Basis data dibuka lewat `buka_db()` agar kunci asing selalu aktif dan koneksi selalu ditutup.
3. Baris yang ditolak *constraint* masuk tabel `tolakan` beserta pesan basis data; jumlah baris direkonsiliasi.
4. Nilai teramati tidak diubah; setiap keputusan dicatat pada `log-keputusan.csv` di akar workspace.
5. Sebelum mengumpulkan: *Kernel → Restart Kernel and Run All Cells*, lalu jalankan Checkpoint.

## Bagian A — Skema relasional 3NF di SQLite

### A.1 Menyiapkan notebook

In [ ]:
import hashlib
import json
import shutil
import sqlite3
import sys
import time
import warnings
from collections import Counter
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

# Kernel DSWorkbench menjalankan sel di luar thread utama, sehingga backend GUI
# (mis. "macosx") gagal membuat figure. Di luar Jupyter/ipykernel dipakai Agg;
# figure yang terbuka tetap ditampilkan kernel sebagai gambar di bawah sel.
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
    warnings.filterwarnings("ignore", message=r"FigureCanvas\w* is non-interactive")
import matplotlib.pyplot as plt

try:
    import mongomock                      # MongoDB tiruan di memori, API sama dengan pymongo
except ImportError as galat:
    raise ImportError(
        "Pustaka mongomock belum ada di lingkungan Python ini. Perbarui aplikasi DSWorkbench / Local Runner ke versi terbaru agar lingkungan python-data-science disegarkan, lalu buka ulang notebook dari halaman Modul 08. "
        "Jangan memasang paket dari dalam notebook.") from galat

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / "raw").is_dir()), None)
if ROOT is None:   # dataset mentah belum disiapkan di laptop ini
    raise FileNotFoundError(
        "Folder data/raw/ (dataset mentah) belum ada di folder data-wrangling/. Di aplikasi "
        "DSWorkbench, nyalakan Local Runner lalu tekan Siapkan berkas modul pada modul ini; "
        "dataset disiapkan ke data/raw/. Setelah itu jalankan ulang sel ini.")
RAW = ROOT / "data" / "raw"
M04 = ROOT / "data" / "acuan"   # keluaran acuan Modul 4 (disediakan)
M07 = ROOT / "data" / "acuan"   # keluaran acuan Modul 7 (disediakan)
MODUL = ROOT / "dw-module-08"      # folder tempat Workbench menyimpan notebook ini
LOG = ROOT / "log-keputusan.csv"
OUT = MODUL / "keluaran"
OUT.mkdir(parents=True, exist_ok=True)
DB = OUT / "lapak_m08.sqlite"
SEED = 2026
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

perlu = ["transaksi_m04.parquet", "produk_m04.parquet", "pelanggan_m04.parquet"]
kurang = [b for b in perlu if not (M04 / b).exists()]
if kurang:
    raise FileNotFoundError(f"Belum ada di {M04}: {', '.join(kurang)}. "
                            "Unduh data-acuan-modul-08.zip di halaman modul dan ekstrak di folder data-wrangling/.")
transaksi = pd.read_parquet(M04 / "transaksi_m04.parquet")
produk_k = pd.read_parquet(M04 / "produk_m04.parquet")
pelanggan_k = pd.read_parquet(M04 / "pelanggan_m04.parquet")
ADA_M07 = (M07 / "toko_m07.parquet").exists()       # Modul 7 opsional
toko_m07 = pd.read_parquet(M07 / "toko_m07.parquet") if ADA_M07 else None
print("Akar repositori :", ROOT.name)
print("transaksi_m04   :", transaksi.shape, "| produk_m04:", produk_k.shape, "| pelanggan_m04:", pelanggan_k.shape)
print("toko_m07        :", toko_m07.shape if ADA_M07 else "tidak ada -> tabel toko dibangun dari katalog saja")
print("SQLite", sqlite3.sqlite_version, "| mongomock", mongomock.__version__, "| pandas", pd.__version__)

In [ ]:
def rp(x: float) -> str:
    """Rupiah berformat Indonesia untuk teks bukti: 1234567 -> 'Rp 1.234.567'."""
    return "Rp " + f"{x:,.0f}".replace(",", ".")


def angka(x: float, d: int = 1) -> str:
    """Angka berformat Indonesia untuk teks bukti: 90.66 -> '90,7'."""
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")


@contextmanager
def buka_db(path: Path | str | None = None, fk: bool = True):
    """Satu koneksi = satu transaksi: commit bila selesai, rollback bila galat, selalu ditutup.
    Kunci asing SQLite HARUS dinyalakan pada setiap koneksi."""
    con = sqlite3.connect(DB if path is None else path)
    con.execute(f"PRAGMA foreign_keys = {'ON' if fk else 'OFF'}")
    try:
        yield con
        con.commit()
    except BaseException:
        con.rollback()
        raise
    finally:
        con.close()


def kueri(sql: str, par: tuple = ()) -> pd.DataFrame:
    """Jalankan SELECT pada lapak_m08.sqlite dan kembalikan DataFrame."""
    with buka_db() as con:
        return pd.read_sql_query(sql, con, params=par)


B: dict = {}       # angka bukti yang dipakai ulang pada tabel, log, dan pembahasan

### A.2 Ketergantungan fungsional: kolom mana milik tabel mana

In [ ]:
def uji_fd(df: pd.DataFrame, tabel: str, penentu: str, tergantung: str) -> dict:
    """Uji ketergantungan fungsional `penentu -> tergantung` pada data.
    n_melanggar       : banyak nilai penentu yang memetakan ke lebih dari satu nilai (kosong diabaikan)
    n_kosong_sebagian : banyak nilai penentu yang sebagian barisnya kosong, sebagian terisi"""
    isi = df.dropna(subset=[penentu])
    n_nilai = isi.groupby(penentu)[tergantung].nunique(dropna=True)
    kosong = isi[tergantung].isna().groupby(isi[penentu]).agg(["any", "all"])
    return {"tabel_sumber": tabel, "penentu": penentu, "tergantung": tergantung,
            "n_penentu": int(n_nilai.size), "n_melanggar": int((n_nilai > 1).sum()),
            "n_kosong_sebagian": int((kosong["any"] & ~kosong["all"]).sum())}


fd = pd.DataFrame([uji_fd(produk_k, "produk_m04", "id_toko", "nama_toko"),
                   uji_fd(transaksi, "transaksi_m04", "id_pesanan", "status")])
fd

In [ ]:
UJI = [(produk_k, "produk_m04", "id_toko", ["nama_toko", "provinsi_toko", "rating_toko"]),
       (produk_k, "produk_m04", "kategori", ["perlu_pendingin"]),
       (transaksi, "transaksi_m04", "kota_tujuan", ["provinsi_tujuan"]),
       (transaksi, "transaksi_m04", "id_pesanan", ["kanal", "id_pelanggan", "waktu_wib", "metode_bayar", "kota_tujuan",
                                                    "kurir", "status", "lama_kirim_hari", "ongkir", "rating_ulasan"]),
       (pelanggan_k, "pelanggan_m04", "kota", ["provinsi"]),
       (pelanggan_k, "pelanggan_m04", "id_entitas", ["no_hp", "kota", "jenis_kelamin", "tahun_lahir",
                                                       "nama", "email", "tanggal_daftar"])]
fd = pd.DataFrame([uji_fd(df, tabel, penentu, t) for df, tabel, penentu, kolom in UJI for t in kolom])

# Keputusan penempatan: kolom yang BERGANTUNG pada penentu pindah ke tabel berkunci penentu itu;
# kolom yang melanggar tetap di tabel yang butirnya lebih halus.
TUJUAN = {("id_toko", "nama_toko"): "toko", ("id_toko", "provinsi_toko"): "produk (provinsi_asal)",
          ("id_toko", "rating_toko"): "produk (rating_katalog)", ("kategori", "perlu_pendingin"): "kategori",
          ("kota_tujuan", "provinsi_tujuan"): "wilayah", ("kota", "provinsi"): "wilayah",
          ("id_pesanan", "ongkir"): "baris_pesanan", ("id_pesanan", "rating_ulasan"): "baris_pesanan",
          ("id_entitas", "nama"): "akun", ("id_entitas", "email"): "akun", ("id_entitas", "tanggal_daftar"): "akun"}
fd["tabel_tujuan"] = [TUJUAN.get((p, t), {"id_pesanan": "pesanan", "id_entitas": "pelanggan"}.get(p, "-"))
                      for p, t in zip(fd.penentu, fd.tergantung)]
fd.to_csv(OUT / "ketergantungan_m08.csv", index=False)
cari = lambda p, t: fd[(fd.penentu == p) & (fd.tergantung == t)].iloc[0]   # noqa: E731
B["n_toko_katalog"] = int(produk_k.id_toko.nunique())
B["toko_multi_provinsi"] = int(cari("id_toko", "provinsi_toko").n_melanggar)
B["toko_multi_rating"] = int(cari("id_toko", "rating_toko").n_melanggar)
B["pesanan_multi_ongkir"] = int(cari("id_pesanan", "ongkir").n_melanggar)
B["pesanan_rating_sebagian"] = int(cari("id_pesanan", "rating_ulasan").n_kosong_sebagian)
B["pesanan_waktu_sebagian"] = int(cari("id_pesanan", "waktu_wib").n_kosong_sebagian)
B["pesanan_lama_sebagian"] = int(cari("id_pesanan", "lama_kirim_hari").n_kosong_sebagian)
B["entitas_multi_nama"] = int(cari("id_entitas", "nama").n_melanggar)
B["entitas_multi_email"] = int(cari("id_entitas", "email").n_melanggar)
B["entitas_multi_tanggal"] = int(cari("id_entitas", "tanggal_daftar").n_melanggar)
# provinsi_toko ternyata mengikuti wilayah pada nama produk: ia atribut PRODUK (asal barang)
wilayah_nama = produk_k.nama_produk.str.extract(r"^(?:" + "|".join(sorted(produk_k.kategori.unique(), key=len, reverse=True))
                                                + r") (.+) \d+$")[0]
B["wilayah_nama_fd"] = int(produk_k.groupby(wilayah_nama).provinsi_toko.nunique().gt(1).sum())
print(f"provinsi berbeda per kata wilayah pada nama produk: {B['wilayah_nama_fd']} "
      "(Riau dan Kepulauan Riau sama-sama ditulis 'Riau')")
fd

### A.3 DDL: kunci primer, kunci asing, `CHECK`, dan `NOT NULL`

In [ ]:
SKEMA_SQL = """
CREATE TABLE wilayah (
    kota      TEXT PRIMARY KEY,
    provinsi  TEXT NOT NULL
);
CREATE TABLE kategori (
    kategori         TEXT PRIMARY KEY,
    perlu_pendingin  INTEGER NOT NULL CHECK (perlu_pendingin IN (0, 1))
);
CREATE TABLE toko (
    id_toko        TEXT PRIMARY KEY,
    nama_toko      TEXT NOT NULL,
    kota           TEXT REFERENCES wilayah (kota),
    provinsi       TEXT,
    rating_toko    REAL CHECK (rating_toko IS NULL OR rating_toko BETWEEN 1 AND 5),
    status         TEXT,
    sumber_profil  TEXT NOT NULL CHECK (sumber_profil IN ('api', 'web', 'katalog'))
);
CREATE TABLE produk (
    kode_produk       TEXT PRIMARY KEY,
    nama_produk       TEXT NOT NULL UNIQUE,
    id_toko           TEXT NOT NULL REFERENCES toko (id_toko),
    kategori          TEXT NOT NULL REFERENCES kategori (kategori),
    harga_katalog_rp  INTEGER NOT NULL CHECK (harga_katalog_rp > 0),
    berat_gram        INTEGER NOT NULL CHECK (berat_gram > 0),
    provinsi_asal     TEXT NOT NULL,
    rating_katalog    REAL CHECK (rating_katalog IS NULL OR rating_katalog BETWEEN 1 AND 5)
);
CREATE TABLE pelanggan (                -- satu baris = satu orang (entitas Modul 4)
    id_entitas     TEXT PRIMARY KEY,
    no_hp          TEXT NOT NULL UNIQUE,
    kota           TEXT NOT NULL REFERENCES wilayah (kota),
    jenis_kelamin  TEXT CHECK (jenis_kelamin IS NULL OR jenis_kelamin IN ('L', 'P')),
    tahun_lahir    INTEGER CHECK (tahun_lahir IS NULL OR tahun_lahir BETWEEN 1900 AND 2026)
);
CREATE TABLE akun (                     -- satu baris = satu id sumber (web atau aplikasi)
    id_pelanggan    TEXT PRIMARY KEY,
    id_entitas      TEXT NOT NULL REFERENCES pelanggan (id_entitas),
    sumber          TEXT NOT NULL CHECK (sumber IN ('web', 'app')),
    nama            TEXT NOT NULL,
    email           TEXT NOT NULL,
    tanggal_daftar  TEXT NOT NULL,
    duplikat        INTEGER NOT NULL CHECK (duplikat IN (0, 1))
);
CREATE TABLE pesanan (
    id_pesanan       TEXT PRIMARY KEY,
    kanal            TEXT NOT NULL CHECK (kanal IN ('web', 'app')),
    id_pelanggan     TEXT REFERENCES akun (id_pelanggan),          -- NULL = pembeli tamu
    waktu_pesan      TEXT,                                          -- WIB, 'YYYY-MM-DD HH:MM:SS'
    metode_bayar     TEXT NOT NULL CHECK (metode_bayar IN ('E-Wallet', 'Transfer Bank', 'COD', 'Kartu Kredit', 'Paylater')),
    kota_tujuan      TEXT NOT NULL REFERENCES wilayah (kota),
    kurir            TEXT NOT NULL,
    status           TEXT NOT NULL CHECK (status IN ('selesai', 'diproses', 'dibatalkan', 'dikembalikan')),
    lama_kirim_hari  INTEGER CHECK (lama_kirim_hari IS NULL OR lama_kirim_hari >= 0)
);
CREATE TABLE baris_pesanan (
    id_pesanan     TEXT NOT NULL REFERENCES pesanan (id_pesanan),
    baris_ke       INTEGER NOT NULL CHECK (baris_ke >= 1),
    kode_produk    TEXT NOT NULL REFERENCES produk (kode_produk),
    jumlah         INTEGER NOT NULL CHECK (jumlah > 0),
    harga_satuan   INTEGER NOT NULL CHECK (harga_satuan > 0),
    diskon_persen  REAL NOT NULL CHECK (diskon_persen BETWEEN 0 AND 100),
    ongkir         INTEGER NOT NULL CHECK (ongkir >= 0),
    total_bayar    INTEGER NOT NULL CHECK (total_bayar >= 0),
    rating_ulasan  INTEGER CHECK (rating_ulasan IS NULL OR rating_ulasan BETWEEN 1 AND 5),
    PRIMARY KEY (id_pesanan, baris_ke)
);
CREATE TABLE tolakan (                  -- baris yang ditolak constraint, beserta pesan basis data
    tabel     TEXT NOT NULL,
    kunci     TEXT NOT NULL,
    aturan    TEXT NOT NULL,
    pesan_db  TEXT NOT NULL,
    isi       TEXT NOT NULL,                -- baris asli sebagai JSON
    PRIMARY KEY (tabel, kunci)
);
"""


def buat_skema(path: Path | str | None = None) -> None:
    """Bangun ulang basis data dari nol: notebook aman dijalankan berulang."""
    if path is None:
        DB.unlink(missing_ok=True)
    with buka_db(path) as con:
        con.executescript(SKEMA_SQL)


buat_skema()
with buka_db() as con:
    for nama, in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        kolom = [f"{k[1]}{'*' if k[5] else ''}" for k in con.execute(f"PRAGMA table_info({nama})")]
        asing = [f"{k[3]}->{k[2]}" for k in con.execute(f"PRAGMA foreign_key_list({nama})")]
        print(f"{nama:<14} {', '.join(kolom)}" + (f"\n{'':<14} FK: {', '.join(sorted(asing))}" if asing else ""))

### A.4 Memuat data dan membiarkan *constraint* menolak

In [ ]:
def bulat(s: pd.Series) -> pd.Series:
    """Rupiah disimpan sebagai INTEGER agar jumlahnya eksak; berhenti bila ada pecahan."""
    assert (s.dropna() % 1 == 0).all(), f"{s.name}: ada nilai pecahan"
    return s.astype("Int64")


def siapkan_toko() -> pd.DataFrame:
    """Profil toko dari Modul 7 bila ada; toko katalog yang tidak ada di sana tetap dibuatkan baris."""
    katalog = (produk_k[["id_toko", "nama_toko"]].drop_duplicates().sort_values("id_toko")
               .assign(kota=None, provinsi=None, rating_toko=np.nan, status=None, sumber_profil="katalog"))
    if not ADA_M07:
        return katalog.reset_index(drop=True)
    m07 = toko_m07[["id_toko", "nama_toko", "kota", "provinsi", "rating_toko", "status", "sumber_profil"]]
    return pd.concat([m07, katalog[~katalog.id_toko.isin(m07.id_toko)]]).sort_values("id_toko").reset_index(drop=True)


T = {}                                          # nama tabel -> DataFrame siap muat
T["wilayah"] = (pd.concat([transaksi[["kota_tujuan", "provinsi_tujuan"]].set_axis(["kota", "provinsi"], axis=1),
                           pelanggan_k[["kota", "provinsi"]]])
                .drop_duplicates().sort_values("kota").reset_index(drop=True))
T["kategori"] = produk_k[["kategori", "perlu_pendingin"]].drop_duplicates().sort_values("kategori").reset_index(drop=True)
T["toko"] = siapkan_toko()
T["produk"] = (produk_k[["kode_produk", "nama_produk", "id_toko", "kategori", "harga_katalog_rp", "berat_gram",
                         "provinsi_toko", "rating_toko"]]
               .rename(columns={"provinsi_toko": "provinsi_asal", "rating_toko": "rating_katalog"}))

per_entitas = pelanggan_k.sort_values("id_pelanggan").groupby("id_entitas")
T["pelanggan"] = per_entitas[["no_hp", "kota", "jenis_kelamin", "tahun_lahir"]].first().reset_index()   # nilai tidak kosong pertama
T["akun"] = (pelanggan_k[["id_pelanggan", "id_entitas", "sumber", "nama", "email", "tanggal_daftar", "duplikat"]]
             .assign(tanggal_daftar=lambda d: d.tanggal_daftar.dt.strftime("%Y-%m-%d %H:%M:%S"))
             .sort_values("id_pelanggan").reset_index(drop=True))

per_pesanan = transaksi.groupby("id_pesanan", sort=True)
T["pesanan"] = (per_pesanan[["kanal", "id_pelanggan", "metode_bayar", "kota_tujuan", "kurir", "status"]].first()
                .assign(waktu_pesan=per_pesanan.waktu_wib.min().dt.strftime("%Y-%m-%d %H:%M:%S"),
                        lama_kirim_hari=per_pesanan.lama_kirim_hari.max())
                .reset_index()[["id_pesanan", "kanal", "id_pelanggan", "waktu_pesan", "metode_bayar",
                                "kota_tujuan", "kurir", "status", "lama_kirim_hari"]])

T["baris_pesanan"] = (transaksi[["id_pesanan", "baris_ke", "kode_produk", "jumlah", "harga_satuan", "diskon_persen",
                                 "ongkir", "total_bayar", "rating_ulasan"]]
                      .assign(baris_ke=lambda d: d.baris_ke.astype(int), harga_satuan=lambda d: bulat(d.harga_satuan),
                              ongkir=lambda d: bulat(d.ongkir), total_bayar=lambda d: bulat(d.total_bayar)))

for nama, df in T.items():
    print(f"{nama:<14} {len(df):>6} baris x {df.shape[1]} kolom")

In [ ]:
KUNCI = {"wilayah": ["kota"], "kategori": ["kategori"], "toko": ["id_toko"], "produk": ["kode_produk"],
         "pelanggan": ["id_entitas"], "akun": ["id_pelanggan"], "pesanan": ["id_pesanan"],
         "baris_pesanan": ["id_pesanan", "baris_ke"], "ulasan": ["id_ulasan"]}
URUTAN = ["wilayah", "kategori", "toko", "produk", "pelanggan", "akun", "pesanan", "baris_pesanan"]   # induk dulu


def rekaman(df: pd.DataFrame) -> list[tuple]:
    """DataFrame -> daftar tuple bertipe Python asli; NaN/NA/NaT menjadi None (NULL)."""
    obj = df.astype(object).where(df.notna(), None)
    return [tuple(x.item() if isinstance(x, np.generic) else x for x in r)
            for r in obj.itertuples(index=False, name=None)]


def aturan_dilanggar(con: sqlite3.Connection, tabel: str, isi: dict, pesan: str) -> str:
    """Terjemahkan pesan IntegrityError menjadi aturan yang dilanggar. Untuk kunci asing SQLite
    tidak menyebut kunci MANA yang gagal, jadi setiap induk diperiksa satu per satu."""
    if not pesan.startswith("FOREIGN KEY"):
        return pesan.replace(" constraint failed", "")           # mis. "NOT NULL: baris_pesanan.jumlah"
    gagal = []
    for _, _, induk, kolom, kolom_induk, *_ in con.execute(f"PRAGMA foreign_key_list({tabel})"):
        nilai = isi.get(kolom)
        if nilai is not None and not con.execute(
                f"SELECT 1 FROM {induk} WHERE {kolom_induk} = ?", (nilai,)).fetchone():
            gagal.append(f"FK {kolom} -> {induk}")
    return "; ".join(sorted(gagal)) or pesan


def muat(con: sqlite3.Connection, tabel: str, df: pd.DataFrame) -> tuple[int, int]:
    """Sisipkan baris demi baris. Baris yang ditolak constraint TIDAK dibuang diam-diam:
    ia dicatat pada tabel `tolakan` beserta pesan basis data. Mengembalikan (masuk, ditolak)."""
    kolom = list(df.columns)
    sql = f"INSERT INTO {tabel} ({', '.join(kolom)}) VALUES ({', '.join('?' * len(kolom))})"
    masuk = ditolak = 0
    for r in rekaman(df):
        try:
            con.execute(sql, r)
            masuk += 1
        except sqlite3.IntegrityError as galat:
            isi = dict(zip(kolom, r))
            con.execute("INSERT INTO tolakan VALUES (?, ?, ?, ?, ?)",
                        (tabel, "|".join(str(isi[k]) for k in KUNCI[tabel]),
                         aturan_dilanggar(con, tabel, isi, str(galat)), str(galat),
                         json.dumps(isi, ensure_ascii=False)))
            ditolak += 1
    return masuk, ditolak


def muat_semua(con: sqlite3.Connection) -> pd.DataFrame:
    hasil = []
    for tabel in URUTAN:
        masuk, ditolak = muat(con, tabel, T[tabel])
        hasil.append({"tabel": tabel, "baris_sumber": len(T[tabel]), "masuk": masuk, "ditolak": ditolak})
    return pd.DataFrame(hasil)


# Percobaan 1: skema yang sama di memori, TANPA menyalakan kunci asing.
with buka_db(":memory:", fk=False) as con:
    con.executescript(SKEMA_SQL)
    tanpa_fk = muat_semua(con)
    yatim = con.execute("PRAGMA foreign_key_check").fetchall()      # (tabel, rowid, induk, indeks FK)
print("tanpa PRAGMA foreign_keys:")
print(tanpa_fk.to_string(index=False))
print("baris yang melanggar kunci asing tetapi tetap masuk:", len(yatim))

In [ ]:
# Percobaan 2: basis data sungguhan, kunci asing menyala (bawaan buka_db).
buat_skema()
with buka_db() as con:
    hasil_muat = muat_semua(con)
    tolakan = pd.read_sql_query("SELECT * FROM tolakan ORDER BY tabel, kunci", con)
    kosong = pd.read_sql_query(
        "SELECT p.id_pesanan, p.status FROM pesanan p "
        "WHERE NOT EXISTS (SELECT 1 FROM baris_pesanan b WHERE b.id_pesanan = p.id_pesanan)", con)
    total_db = con.execute("SELECT SUM(total_bayar) FROM baris_pesanan").fetchone()[0]
assert (hasil_muat.baris_sumber == hasil_muat.masuk + hasil_muat.ditolak).all()
hasil_muat.to_csv(OUT / "muat_m08.csv", index=False)
tolakan.to_csv(OUT / "tolakan_m08.csv", index=False)

isi_tolak = pd.DataFrame([json.loads(x) for x in tolakan.isi])
total_tolak = int(isi_tolak.total_bayar.sum())
assert total_db + total_tolak == int(transaksi.total_bayar.sum())        # tidak ada rupiah yang hilang
status_tolak = transaksi.set_index(["id_pesanan", transaksi.baris_ke.astype(int)]).status.reindex(
    pd.MultiIndex.from_frame(isi_tolak[["id_pesanan", "baris_ke"]])).to_numpy()
B.update(baris_sumber=len(transaksi), baris_masuk=int(hasil_muat.set_index("tabel").masuk["baris_pesanan"]),
         baris_tolak=len(tolakan), tolak_fk=int(tolakan.aturan.str.startswith("FK").sum()),
         tolak_null=int(tolakan.aturan.str.startswith("NOT NULL").sum()),
         yatim_tanpa_pragma=len(yatim), masuk_tanpa_pragma=int(tanpa_fk.set_index("tabel").masuk["baris_pesanan"]),
         pesanan=len(T["pesanan"]), pesanan_kosong=len(kosong), total_tolak=total_tolak,
         total_tolak_selesai=int(isi_tolak.total_bayar[status_tolak == "selesai"].sum()),
         tolak_selesai=int((status_tolak == "selesai").sum()), total_db=int(total_db))
print(hasil_muat.to_string(index=False))
print("\naturan yang menolak:", tolakan.groupby(["tabel", "aturan"]).size().to_dict())
print(f"nilai baris ditolak: {rp(total_tolak)} ({B['tolak_selesai']} baris selesai, {rp(B['total_tolak_selesai'])})")
print(f"pesanan tanpa baris sesudah penolakan: {len(kosong)} dari {B['pesanan']}")
tolakan.head(3)

**Pembahasan A**

| Ketergantungan yang diuji | Melanggar | Keputusan |
|---|---:|---|
| `id_toko → nama_toko` | 0 dari 177 | `toko` |
| `id_toko → provinsi_toko` | **166** | bukan atribut toko → `produk.provinsi_asal` |
| `id_toko → rating_toko` | **161** | bukan atribut toko → `produk.rating_katalog` |
| `kategori → perlu_pendingin` | 0 dari 7 | tabel `kategori` (ketergantungan transitif) |
| `kota → provinsi` (tujuan dan pelanggan) | 0 dari 10 | tabel `wilayah` |
| `id_pesanan → ongkir` | **3.357** | `baris_pesanan` |
| `id_pesanan → waktu_wib`, `lama_kirim_hari` | 0 (kosong sebagian 10 dan 81) | `pesanan` |
| `id_pesanan → rating_ulasan` | 0 (kosong sebagian **2.434**) | tetap `baris_pesanan` (lihat di bawah) |
| `id_entitas → nama`, `email`, `tanggal_daftar` | 159, 220, 700 | `akun` |
| `id_entitas → no_hp`, `kota`, `jenis_kelamin`, `tahun_lahir` | 0 | `pelanggan` |

- **`provinsi_toko` dan `rating_toko` bukan atribut toko.** 166 dari 177 toko tercatat di lebih
  dari satu provinsi di antara produknya. Menaruhnya di tabel `toko` berarti memilih satu nilai
  dan membuang sisanya tanpa catatan. Kolom itu mengikuti kata wilayah pada `nama_produk`
  (asal barang), sehingga tempatnya di `produk` sebagai `provinsi_asal`; profil toko yang
  sebenarnya datang dari Modul 7. Nama kolom tidak membuktikan butirnya; yang membuktikan adalah
  uji ketergantungan.
- **"Melanggar" berbeda dengan "kosong sebagian".** `waktu_wib` dan `lama_kirim_hari` tidak pernah
  bernilai dua dalam satu pesanan, tetapi kosong pada sebagian barisnya (10 dan 81 pesanan). Itu
  atribut pesanan yang direkam berulang di setiap baris: sesudah dinormalkan, satu nilai yang
  terisi cukup, dan redundansi yang hilang sekaligus menghilangkan ketidakkonsistenan itu.
- **Ketergantungan yang terpenuhi di data belum tentu aturan.** `rating_ulasan` juga tidak pernah
  bernilai dua dalam satu pesanan, dan kosong sebagian pada 2.434 pesanan. Uji yang sama, tetapi
  keputusannya berbeda: kosong pada waktu adalah kegagalan pencatatan satu peristiwa, sedangkan
  kosong pada rating berarti pembeli **tidak menilai** barang itu (Modul 2). Memindahkannya ke
  `pesanan` akan mengisi baris yang tidak pernah dinilai. Uji ketergantungan adalah syarat perlu;
  makna kolomnya yang memutuskan.
- **Entitas dan akun dipisah.** Nama, email, dan tanggal daftar berbeda antar-id dalam satu
  entitas (159, 220, dan 700 entitas), sehingga mereka fakta tentang **akun**. Nomor HP, kota,
  jenis kelamin, dan tahun lahir tidak pernah bertentangan, sehingga menjadi fakta tentang
  **orang**. `pesanan` merujuk `akun` (id yang benar-benar tertulis di transaksi), dan orangnya
  dicapai lewat `akun.id_entitas`.
- **Kunci asing SQLite mati secara bawaan, per koneksi.** Tanpa `PRAGMA foreign_keys = ON`, skema
  yang sama menerima **23.999** baris pesanan: hanya `NOT NULL` yang bekerja, dan **76** baris
  berkode produk yatim masuk tanpa galat (`PRAGMA foreign_key_check` menemukannya belakangan).
  Deklarasi `REFERENCES` tanpa pragma itu hanyalah dokumentasi.
- **Dengan kunci asing menyala: 24.000 = 23.923 masuk + 77 ditolak.** 76 baris ditolak
  `FK kode_produk -> produk` (kode `LS-UNK-9xxx`, A-16) dan 1 baris ditolak
  `NOT NULL: baris_pesanan.jumlah` (jumlah yang dikosongkan Modul 3). Pesan SQLite untuk kunci
  asing tidak menyebut kunci yang gagal; `aturan_dilanggar()` mencarinya lewat
  `PRAGMA foreign_key_list`.
- **Penolakan bukan akhir cerita.** Baris yang ditolak bernilai Rp 23.343.100 (60 di antaranya
  berstatus selesai, Rp 18.276.600) dan **39 pesanan kini tidak memiliki baris sama sekali**.
  Kepala pesanan itu tetap disimpan karena faktanya sah; yang hilang dari laporan omzet adalah
  nilainya, dan itu tercatat di `tolakan`. Jumlah `total_bayar` basis data ditambah tolakan sama
  persis dengan `transaksi_m04`.
- **Penyimpangan sadar dari 3NF:** `toko.provinsi` disimpan di samping `toko.kota` walaupun
  `kota → provinsi`, karena satu toko (hanya ada di katalog) memiliki provinsi tanpa kota.
  Konsistensinya diuji dengan kueri, bukan diasumsikan.

## Bagian B — Kueri, pembanding pandas, dan rencana kueri

### B.1 JOIN dan GROUP BY: omzet per provinsi

In [ ]:
Q_OMZET = """
SELECT w.provinsi,
       COUNT(DISTINCT p.id_pesanan) AS n_pesanan,
       SUM(b.jumlah)                AS unit,
       SUM(b.total_bayar)           AS omzet
FROM baris_pesanan b
JOIN pesanan p ON p.id_pesanan = b.id_pesanan
JOIN wilayah w ON w.kota = p.kota_tujuan
WHERE p.status = 'selesai'
GROUP BY w.provinsi
ORDER BY w.provinsi
"""
omzet_sql = kueri(Q_OMZET)

selesai = transaksi[transaksi.status.eq("selesai")]
omzet_pd = (selesai.groupby("provinsi_tujuan")
            .agg(n_pesanan=("id_pesanan", "nunique"), unit=("jumlah", "sum"), omzet=("total_bayar", "sum"))
            .rename_axis("provinsi").reset_index())
banding = omzet_sql.merge(omzet_pd, on="provinsi", suffixes=("_sql", "_pandas"))
banding["selisih_omzet"] = banding.omzet_pandas - banding.omzet_sql
banding[["provinsi", "n_pesanan_sql", "n_pesanan_pandas", "omzet_sql", "omzet_pandas", "selisih_omzet"]]

In [ ]:
HASIL_KUERI: list[dict] = []


def bandingkan(nama: str, pembanding: str, sql_df: pd.DataFrame, pd_df: pd.DataFrame, kolom: str) -> None:
    """Catat apakah hasil SQL dan pandas sama, dan berapa selisih total kolom ukurannya."""
    sama = len(sql_df) == len(pd_df) and all(
        np.array_equal(sql_df[k].to_numpy(), pd_df[k].to_numpy()) for k in sql_df.columns)
    HASIL_KUERI.append({"kueri": nama, "pembanding": pembanding, "n_baris": len(sql_df), "sama": bool(sama),
                        "selisih": int(pd_df[kolom].sum() - sql_df[kolom].sum())})


def agregat_pd(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby("provinsi_tujuan")
            .agg(n_pesanan=("id_pesanan", "nunique"), unit=("jumlah", "sum"), omzet=("total_bayar", "sum"))
            .rename_axis("provinsi").reset_index().astype({"unit": "int64", "omzet": "int64"}))


kunci_tolak = set(zip(isi_tolak.id_pesanan, isi_tolak.baris_ke))
diterima = transaksi[[k not in kunci_tolak for k in zip(transaksi.id_pesanan, transaksi.baris_ke.astype(int))]]
bandingkan("omzet_provinsi", "pandas: semua baris transaksi_m04", omzet_sql, agregat_pd(selesai.assign(jumlah=selesai.jumlah.fillna(0))), "omzet")
bandingkan("omzet_provinsi", "pandas: tanpa baris yang ditolak", omzet_sql, agregat_pd(diterima[diterima.status.eq("selesai")]), "omzet")
B["selisih_omzet"] = HASIL_KUERI[0]["selisih"]
assert B["selisih_omzet"] == B["total_tolak_selesai"] and HASIL_KUERI[1]["sama"]
print(pd.DataFrame(HASIL_KUERI).to_string(index=False))
print(f"\nselisih {rp(B['selisih_omzet'])} = nilai {B['tolak_selesai']} baris 'selesai' yang ditolak constraint")

### B.2 Window function: tiga produk teratas per provinsi

In [ ]:
Q_PERINGKAT = """
WITH jual AS (
    SELECT w.provinsi, b.kode_produk, SUM(b.jumlah) AS unit, SUM(b.total_bayar) AS omzet
    FROM baris_pesanan b
    JOIN pesanan p ON p.id_pesanan = b.id_pesanan
    JOIN wilayah w ON w.kota = p.kota_tujuan
    WHERE p.status = 'selesai'
    GROUP BY w.provinsi, b.kode_produk
), urut AS (
    SELECT *, RANK() OVER (PARTITION BY provinsi ORDER BY omzet DESC) AS peringkat
    FROM jual
)
SELECT u.provinsi, u.peringkat, u.kode_produk, pr.nama_produk, k.kategori, t.nama_toko, u.unit, u.omzet
FROM urut u
JOIN produk pr  ON pr.kode_produk = u.kode_produk
JOIN kategori k ON k.kategori = pr.kategori
JOIN toko t     ON t.id_toko = pr.id_toko
WHERE u.peringkat <= 3
ORDER BY u.provinsi, u.peringkat, u.kode_produk
"""
peringkat_sql = kueri(Q_PERINGKAT)
peringkat_sql.head(6)

In [ ]:
jual = (diterima[diterima.status.eq("selesai")]
        .groupby(["provinsi_tujuan", "kode_produk"]).agg(unit=("jumlah", "sum"), omzet=("total_bayar", "sum"))
        .reset_index().rename(columns={"provinsi_tujuan": "provinsi"}))
jual["peringkat"] = jual.groupby("provinsi").omzet.rank(method="min", ascending=False).astype(int)
peringkat_pd = (jual[jual.peringkat <= 3].sort_values(["provinsi", "peringkat", "kode_produk"])
                .astype({"unit": "int64", "omzet": "int64"}).reset_index(drop=True))
bandingkan("peringkat_produk", "pandas: tanpa baris yang ditolak",
           peringkat_sql[["provinsi", "peringkat", "kode_produk", "unit", "omzet"]],
           peringkat_pd[["provinsi", "peringkat", "kode_produk", "unit", "omzet"]], "omzet")
assert HASIL_KUERI[-1]["sama"]
pd.DataFrame(HASIL_KUERI).to_csv(OUT / "kueri_m08.csv", index=False)
B["n_peringkat"] = len(peringkat_sql)
print(pd.DataFrame(HASIL_KUERI).to_string(index=False))

### B.3 `EXPLAIN QUERY PLAN`, indeks, dan waktu kueri

In [ ]:
def rencana(sql: str, par: tuple = ()) -> str:
    """Rencana kueri SQLite sebagai satu baris teks."""
    with buka_db() as con:
        return " | ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql, par))


def waktu_ms(sql: str, par: tuple = (), ulang: int = 30) -> tuple[float, int]:
    """(median waktu dalam milidetik, banyak baris hasil) dari `ulang` kali jalan."""
    lama = []
    with buka_db() as con:
        for _ in range(ulang):
            t0 = time.perf_counter()
            n = len(con.execute(sql, par).fetchall())
            lama.append((time.perf_counter() - t0) * 1000)
    return float(np.median(lama)), n


laris = transaksi.kode_produk[transaksi.produk_dikenal].value_counts().index[0]
setia = transaksi.id_pelanggan.value_counts().index[0]
KUERI_UJI = {       # nama -> (SQL, parameter)
    "baris_per_produk": ("SELECT id_pesanan, baris_ke, jumlah, total_bayar FROM baris_pesanan WHERE kode_produk = ?", (laris,)),
    "pesanan_per_pelanggan": ("SELECT id_pesanan, status FROM pesanan WHERE id_pelanggan = ?", (setia,)),
    "bulan_fungsi": ("SELECT id_pesanan FROM pesanan WHERE substr(waktu_pesan, 1, 7) = ?", ("2026-03",)),
    "bulan_rentang": ("SELECT id_pesanan FROM pesanan WHERE waktu_pesan >= ? AND waktu_pesan < ?", ("2026-03-01", "2026-04-01")),
    "baris_per_pesanan": ("SELECT * FROM baris_pesanan WHERE id_pesanan = ?", (transaksi.id_pesanan.iloc[0],)),
}
INDEKS = ["CREATE INDEX IF NOT EXISTS ix_baris_produk ON baris_pesanan (kode_produk)",
          "CREATE INDEX IF NOT EXISTS ix_pesanan_pelanggan ON pesanan (id_pelanggan)",
          "CREATE INDEX IF NOT EXISTS ix_pesanan_waktu ON pesanan (waktu_pesan)"]


def ukur(kondisi: str) -> list[dict]:
    baris = []
    for nama, (sql, par) in KUERI_UJI.items():
        ms, n = waktu_ms(sql, par)
        baris.append({"kueri": nama, "kondisi": kondisi, "rencana": rencana(sql, par), "n_baris": n, "waktu_ms": round(ms, 4)})
    return baris


def ukuran_db() -> int:
    with buka_db() as con:
        return con.execute("PRAGMA page_count").fetchone()[0] * con.execute("PRAGMA page_size").fetchone()[0]


for nama, (sql, par) in KUERI_UJI.items():
    print(f"{nama:<22} {rencana(sql, par)}")

In [ ]:
with buka_db() as con:
    for nama_indeks in ("ix_baris_produk", "ix_pesanan_pelanggan", "ix_pesanan_waktu"):
        con.execute(f"DROP INDEX IF EXISTS {nama_indeks}")           # aman dijalankan ulang
B["ukuran_tanpa_indeks"] = ukuran_db()
ukuran = ukur("tanpa_indeks")
with buka_db() as con:
    for sql in INDEKS:
        con.execute(sql)
    con.execute("ANALYZE")
B["ukuran_dengan_indeks"] = ukuran_db()
ukuran += ukur("dengan_indeks")
rencana_kueri = pd.DataFrame(ukuran).sort_values(["kueri", "kondisi"], ascending=[True, False]).reset_index(drop=True)
rencana_kueri.to_csv(OUT / "rencana_kueri_m08.csv", index=False)

lebar = rencana_kueri.pivot(index="kueri", columns="kondisi", values="waktu_ms").loc[list(KUERI_UJI)]
fig, ax = plt.subplots(figsize=(8.5, 3.8))
y = np.arange(len(lebar))
ax.barh(y - 0.2, lebar.tanpa_indeks, height=0.38, color="#9aa5b1", label="tanpa indeks")
ax.barh(y + 0.2, lebar.dengan_indeks, height=0.38, color="#1f6f8b", label="dengan indeks")
ax.set_yticks(y, lebar.index)
ax.set_xscale("log")
ax.invert_yaxis()
ax.set_xlabel("median waktu per kueri (ms, skala log)")
ax.set_title("Waktu kueri sebelum dan sesudah indeks")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "01_waktu_kueri.png", dpi=110)
plt.show()

r = rencana_kueri.set_index(["kueri", "kondisi"])
B["rencana_fungsi"] = r.loc[("bulan_fungsi", "dengan_indeks"), "rencana"]
B["rencana_rentang"] = r.loc[("bulan_rentang", "dengan_indeks"), "rencana"]
B["n_bulan"] = int(r.loc[("bulan_rentang", "dengan_indeks"), "n_baris"])
assert r.loc[("bulan_fungsi", "dengan_indeks"), "n_baris"] == B["n_bulan"]
print(f"ukuran berkas: {B['ukuran_tanpa_indeks'] / 1024:,.0f} KiB -> {B['ukuran_dengan_indeks'] / 1024:,.0f} KiB")
rencana_kueri[["kueri", "kondisi", "rencana", "n_baris", "waktu_ms"]]

**Pembahasan B**

- **SQL dan pandas tidak sama, dan selisihnya dapat dijelaskan sampai nol.** Omzet `selesai` dari
  basis data lebih kecil **Rp 18.276.600** daripada `transaksi_m04`: tepat nilai 60 baris
  berstatus selesai yang ditolak *constraint* di Bagian A. Begitu pandas menghitung dari baris
  yang sama, kedua hasil identik di semua provinsi. "Harus sama" baru bermakna setelah kedua
  pihak sepakat tentang **baris mana** yang dihitung; angka yang berbeda tanpa penjelasan adalah
  dua laporan omzet yang sama-sama "benar".
- **Window function** memberi peringkat **di dalam** setiap provinsi tanpa menghilangkan baris
  (berbeda dengan `GROUP BY`). `RANK()` setara dengan `rank(method="min")` pandas; hasil 30 baris
  (3 produk × 10 provinsi) identik. `ROW_NUMBER()` akan memutus seri secara sembarang.
- **Kunci asing tidak otomatis berindeks.** `baris_pesanan.kode_produk` dan
  `pesanan.id_pelanggan` dideklarasikan `REFERENCES`, tetapi rencana awalnya `SCAN`: seluruh
  tabel dibaca. Sesudah indeks, rencananya `SEARCH ... USING INDEX` dan waktunya turun satu
  sampai dua orde. Sebaliknya `baris_per_pesanan` sudah `SEARCH` sejak awal karena memakai indeks
  otomatis kunci primer `(id_pesanan, baris_ke)`.
- **Indeks tidak menolong kueri yang membungkus kolomnya dengan fungsi.**
  `substr(waktu_pesan, 1, 7) = '2026-03'` tetap `SCAN` seluruh tabel, sedangkan bentuk rentang `waktu_pesan >= '2026-03-01' AND waktu_pesan < '2026-04-01'`
  memakai `SEARCH ... USING INDEX`. Hasil keduanya sama (1.005 pesanan); yang berbeda adalah
  pekerjaan basis data.
- **Indeks tidak gratis.** Tiga indeks menambah ukuran berkas (angka pada keluaran sel) dan
  memperlambat setiap `INSERT`. Indeks dibuat untuk kueri yang benar-benar sering dijalankan,
  dan dibuktikan dengan rencana kueri, bukan dengan firasat.
- Waktu kueri bergantung pada mesin; yang dinilai adalah **perubahan rencana**, bukan angkanya.

## Bagian C — Dokumen di mongomock: dua versi skema ulasan

### C.1 Memuat dokumen dan menemukan skemanya

In [ ]:
klien = mongomock.MongoClient()            # di memori; pymongo.MongoClient(...) untuk MongoDB sungguhan
kol = klien["lapak"]["ulasan"]
with open(RAW / "ulasan.jsonl", encoding="utf-8") as f:
    DOK = [json.loads(b) for b in f if b.strip()]
kol.insert_many([dict(d) for d in DOK])    # salinan: insert_many menambahkan _id ke dict yang diberikan
print("dokumen dimuat:", kol.count_documents({}), "| ditolak: 0 (koleksi tidak punya skema)")
for versi in (1, 2):
    print(json.dumps(kol.find_one({"versi_skema": versi}, {"_id": 0}), ensure_ascii=False))


def jalur(nilai, awalan: str = ""):
    """Semua jalur field sebuah dokumen beserta tipe nilainya; isi array diberi akhiran []."""
    if isinstance(nilai, dict):
        for k, v in nilai.items():
            if k != "_id":
                yield from jalur(v, f"{awalan}.{k}" if awalan else k)
    elif isinstance(nilai, list):
        yield awalan, "array", nilai
        for v in nilai:
            yield from jalur(v, awalan + "[]")
    else:
        yield awalan, type(nilai).__name__, nilai


bentuk = Counter((d["versi_skema"], tuple(sorted(k for k in d if k != "_id"))) for d in kol.find())
for (versi, kunci_atas), n in sorted(bentuk.items(), key=lambda x: (x[0][0], -x[1])):
    print(f"v{versi}  {n:>5} dokumen  {', '.join(kunci_atas)}")

In [ ]:
hitung: dict = {}
for d in kol.find():
    terlihat, kosong = set(), set()
    for j, tipe, nilai in jalur(d):
        terlihat.add((j, tipe))
        if nilai == "" or nilai == []:
            kosong.add((j, tipe))
    for kunci_ in terlihat:
        h = hitung.setdefault(kunci_, {"n_v1": 0, "n_v2": 0, "n_kosong": 0})
        h[f"n_v{d['versi_skema']}"] += 1
        h["n_kosong"] += kunci_ in kosong
skema = (pd.DataFrame([{"jalur": j, "tipe": t, **h} for (j, t), h in hitung.items()])
         .sort_values(["jalur", "tipe"]).reset_index(drop=True))
skema.to_csv(OUT / "skema_ulasan_m08.csv", index=False)
B.update(n_dok=len(DOK), n_v1=kol.count_documents({"versi_skema": 1}), n_v2=kol.count_documents({"versi_skema": 2}),
         n_bentuk=len(bentuk), n_balasan=kol.count_documents({"balasan_penjual": {"$exists": True}}))
skema

### C.2 Kueri: field bersarang, `$exists`, dan isi array

In [ ]:
KUERI_DOK = {
    "v1: rating <= 2": {"rating": {"$lte": 2}},
    "v2: penilaian.bintang <= 2": {"penilaian.bintang": {"$lte": 2}},
    "ada balasan penjual": {"balasan_penjual": {"$exists": True}},
    "bertag 'lambat'": {"tag": "lambat"},
    "tepat dua media": {"media": {"$size": 2}},
    "v2: aspek pengiriman < bintang": {"versi_skema": 2,
                                        "$expr": {"$lt": ["$penilaian.aspek.pengiriman", "$penilaian.bintang"]}},
}
for nama, saring in KUERI_DOK.items():
    print(f"{kol.count_documents(saring):>5}  {nama:<32} {json.dumps(saring, ensure_ascii=False)}")

In [ ]:
KUERI_DOK.update({
    "bintang <= 2 (kedua versi)": {"$or": [{"rating": {"$lte": 2}}, {"penilaian.bintang": {"$lte": 2}}]},
    "id pelanggan: field tidak ada": {"id_pelanggan": {"$exists": False}, "pelanggan.id": {"$exists": False}},
    "id pelanggan: string kosong": {"$or": [{"id_pelanggan": ""}, {"pelanggan.id": ""}]},
    "v2: kategori produk null": {"versi_skema": 2, "produk.kategori": None},
})
kueri_dok = pd.DataFrame([{"kueri": nama, "filter": json.dumps(saring, ensure_ascii=False),
                           "n": kol.count_documents(saring)} for nama, saring in KUERI_DOK.items()])
kueri_dok.to_csv(OUT / "kueri_dokumen_m08.csv", index=False)
n = kueri_dok.set_index("kueri").n
B.update(rendah_v1=int(n["v1: rating <= 2"]), rendah_v2=int(n["v2: penilaian.bintang <= 2"]),
         rendah=int(n["bintang <= 2 (kedua versi)"]), id_kosong=int(n["id pelanggan: string kosong"]),
         id_hilang=int(n["id pelanggan: field tidak ada"]), kategori_null=int(n["v2: kategori produk null"]))
assert B["rendah"] == B["rendah_v1"] + B["rendah_v2"]
kueri_dok

### C.3 Pipeline agregasi: `$match`, `$unwind`, `$group`

In [ ]:
pipa_tag = [
    {"$match": {"versi_skema": 2}},
    {"$unwind": "$tag"},
    {"$group": {"_id": "$tag", "n": {"$sum": 1}, "rata_bintang": {"$avg": "$penilaian.bintang"}}},
    {"$sort": {"n": -1}},
]
tag = pd.DataFrame(kol.aggregate(pipa_tag)).rename(columns={"_id": "tag"})[["tag", "n", "rata_bintang"]]
print(tag.round(3).to_string(index=False))
print("jumlah n:", int(tag.n.sum()), "| dokumen v2:", kol.count_documents({"versi_skema": 2}))


def coba(operator: str, fungsi, cara_lain: str) -> dict:
    """Jalankan satu operasi; catat apakah mongomock mendukungnya dan apa penggantinya."""
    try:
        fungsi()
        return {"operator": operator, "didukung": True, "pesan": "", "cara_lain": cara_lain}
    except Exception as galat:  # noqa: BLE001
        return {"operator": operator, "didukung": False,
                "pesan": f"{type(galat).__name__}: {str(galat)[:80]}", "cara_lain": cara_lain}


DUKUNGAN = [
    coba("$sortByCount", lambda: list(kol.aggregate([{"$unwind": "$tag"}, {"$sortByCount": "$tag"}])),
         "$group dengan $sum, lalu $sort"),
    coba("$dateFromString", lambda: list(kol.aggregate([{"$limit": 1}, {"$project": {"t": {"$dateFromString": {"dateString": "$dibuat_pada"}}}}])),
         "urai tanggal di pandas sesudah dokumen dibaca; di pipeline pakai $substr untuk bulan"),
]

In [ ]:
pipa_tag_semua = [
    {"$match": {"versi_skema": 2}},
    {"$unwind": {"path": "$tag", "preserveNullAndEmptyArrays": True}},
    {"$group": {"_id": {"$ifNull": ["$tag", "(tanpa tag)"]}, "n": {"$sum": 1},
                "rata_bintang": {"$avg": "$penilaian.bintang"}}},
    {"$sort": {"n": -1, "_id": 1}},
]
tag_semua = pd.DataFrame(kol.aggregate(pipa_tag_semua)).rename(columns={"_id": "tag"})[["tag", "n", "rata_bintang"]]
tag_semua["rata_bintang"] = tag_semua.rata_bintang.round(3)
tag_semua.to_csv(OUT / "tag_m08.csv", index=False)
n_tag = Counter(len(d["tag"]) for d in kol.find({"versi_skema": 2}, {"tag": 1}))
B.update(unwind_baris=int(tag.n.sum()), tanpa_tag=n_tag[0], satu_tag=n_tag[1], dua_tag=n_tag[2],
         rata_v2=round(float(pd.DataFrame(kol.aggregate([{"$match": {"versi_skema": 2}},
                                                         {"$group": {"_id": None, "r": {"$avg": "$penilaian.bintang"}}}])).r[0]), 3))
assert B["unwind_baris"] == B["satu_tag"] + 2 * B["dua_tag"] and int(tag_semua.n.sum()) == B["unwind_baris"] + B["tanpa_tag"]
print(tag_semua.to_string(index=False))
print(f"\nv2: {B['n_v2']} dokumen = {B['tanpa_tag']} tanpa tag + {B['satu_tag']} satu tag + {B['dua_tag']} dua tag;"
      f" $unwind biasa menghasilkan {B['unwind_baris']} baris")

pipa_bintang = [
    {"$project": {"versi": "$versi_skema", "bintang": {"$ifNull": ["$penilaian.bintang", "$rating"]}}},
    {"$group": {"_id": {"versi": "$versi", "bintang": "$bintang"}, "n": {"$sum": 1}}},
]
per_bintang = (pd.DataFrame([{**d["_id"], "n": d["n"]} for d in kol.aggregate(pipa_bintang)])
               .sort_values(["versi", "bintang"]).reset_index(drop=True))
pipa_bulan = [
    {"$group": {"_id": {"bulan": {"$substr": ["$dibuat_pada", 0, 7]}, "versi": "$versi_skema"}, "n": {"$sum": 1}}},
]
per_bulan = (pd.DataFrame([{**d["_id"], "n": d["n"]} for d in kol.aggregate(pipa_bulan)])
             .pivot(index="bulan", columns="versi", values="n").fillna(0).sort_index())
B["bulan_ganti"] = per_bulan.index[per_bulan[2] > 0][0]

fig, (kiri, kanan) = plt.subplots(1, 2, figsize=(11, 3.8))
porsi = per_bintang.pivot(index="bintang", columns="versi", values="n").fillna(0)
porsi = porsi / porsi.sum() * 100
x = np.arange(len(porsi))
kiri.bar(x - 0.2, porsi[1], width=0.38, color="#9aa5b1", label="skema v1")
kiri.bar(x + 0.2, porsi[2], width=0.38, color="#1f6f8b", label="skema v2")
kiri.set_xticks(x, porsi.index)
kiri.set_xlabel("bintang")
kiri.set_ylabel("% dokumen dalam versinya")
kiri.set_title("Sebaran bintang per versi skema")
kiri.legend(frameon=False)
kanan.bar(per_bulan.index, per_bulan[1], color="#9aa5b1", label="skema v1")
kanan.bar(per_bulan.index, per_bulan[2], bottom=per_bulan[1], color="#1f6f8b", label="skema v2")
kanan.set_xticks(range(0, len(per_bulan), 3), per_bulan.index[::3], rotation=45, ha="right")
kanan.set_ylabel("dokumen")
kanan.set_title("Dokumen per bulan `dibuat_pada`")
kanan.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "02_ulasan_versi.png", dpi=110)
plt.show()

uji = klien["lapak"]["uji"]
uji.delete_many({})
uji.insert_one({"a": 1})
DUKUNGAN += [
    coba("$toDate", lambda: list(kol.aggregate([{"$limit": 1}, {"$project": {"t": {"$toDate": "$dibuat_pada"}}}])),
         "urai tanggal di pandas sesudah dokumen dibaca"),
    coba("$round", lambda: list(kol.aggregate([{"$limit": 1}, {"$project": {"r": {"$round": ["$rating", 1]}}}])),
         "bulatkan di pandas, atau $trunc sesudah $multiply"),
    coba("$expr pada jalur yang tidak ada", lambda: kol.count_documents(
        {"$expr": {"$lt": ["$penilaian.aspek.pengiriman", "$penilaian.bintang"]}}),
         "batasi dulu dengan filter versi_skema, atau bungkus dengan $ifNull"),
    coba("$rename ke jalur bertitik (update)", lambda: uji.update_many({}, {"$rename": {"a": "b.c"}}),
         "update_many dengan pipeline [{'$set': {...}}], lalu $unset field lama"),
    coba("$lookup", lambda: list(uji.aggregate([{"$lookup": {"from": "uji", "localField": "a", "foreignField": "a", "as": "x"}}])),
         "didukung tetapi tanpa indeks: lambat pada koleksi besar; gabungkan di sisi aplikasi"),
    coba("cursor.explain()", lambda: kol.find({"versi_skema": 1}).explain(),
         "tidak ada rencana kueri di mongomock; ukur waktu dengan time.perf_counter"),
]
dukungan = pd.DataFrame(DUKUNGAN)
dukungan.to_csv(OUT / "mongomock_m08.csv", index=False)
B["tak_didukung"] = int((~dukungan.didukung).sum())
dukungan

### C.4 *Schema-on-read*: pandangan terpadu v1 + v2

In [ ]:
KOLOM_TERPADU = ["id_ulasan", "versi_skema", "id_pesanan", "kode_produk", "id_pelanggan", "bintang",
                 "aspek_kualitas", "aspek_pengiriman", "teks", "n_media", "n_tag", "tag", "ada_balasan",
                 "dibalas_pada", "dibuat_pada", "kanal_dok", "kategori_dok"]


def terpadu(d: dict) -> dict:
    """Satu dokumen (skema v1 atau v2) -> satu baris berbentuk tetap. Yang tidak direkam sebuah
    versi menjadi None (tidak diketahui), BUKAN nol atau False; versi yang belum dikenal menghentikan notebook."""
    versi = d.get("versi_skema")
    if versi == 1:
        return {"id_ulasan": d["id_ulasan"], "versi_skema": 1, "id_pesanan": d["id_pesanan"],
                "kode_produk": d["kode_produk"], "id_pelanggan": d["id_pelanggan"] or None,
                "bintang": d["rating"], "aspek_kualitas": None, "aspek_pengiriman": None, "teks": d["teks"],
                "n_media": len(d["foto"]), "n_tag": None, "tag": None, "ada_balasan": None,
                "dibalas_pada": None, "dibuat_pada": d["dibuat_pada"], "kanal_dok": None, "kategori_dok": None}
    if versi == 2:
        aspek = d["penilaian"].get("aspek", {})
        balasan = d.get("balasan_penjual")
        return {"id_ulasan": d["id_ulasan"], "versi_skema": 2, "id_pesanan": d["pesanan"]["id"],
                "kode_produk": d["produk"]["kode"], "id_pelanggan": d["pelanggan"]["id"] or None,
                "bintang": d["penilaian"]["bintang"], "aspek_kualitas": aspek.get("kualitas"),
                "aspek_pengiriman": aspek.get("pengiriman"), "teks": d["teks"], "n_media": len(d.get("media", [])),
                "n_tag": len(d["tag"]) if "tag" in d else None, "tag": ";".join(d.get("tag", [])) or None,
                "ada_balasan": balasan is not None, "dibalas_pada": balasan["dibalas_pada"] if balasan else None,
                "dibuat_pada": d["dibuat_pada"], "kanal_dok": d["pesanan"].get("kanal"),
                "kategori_dok": d["produk"].get("kategori")}
    raise ValueError(f"{d.get('id_ulasan')}: versi_skema {versi!r} belum dikenal")


def bingkai_terpadu(dokumen) -> pd.DataFrame:
    """Daftar dokumen -> DataFrame bertipe tetap (Tabel kontrak naskah)."""
    df = pd.DataFrame([terpadu(d) for d in dokumen], columns=KOLOM_TERPADU)
    for k in ["bintang", "aspek_kualitas", "aspek_pengiriman", "n_media", "n_tag"]:
        df[k] = df[k].astype("Int64")
    for k in ["id_ulasan", "id_pesanan", "kode_produk", "id_pelanggan", "teks", "tag", "kanal_dok", "kategori_dok"]:
        df[k] = df[k].astype("str")
    df["versi_skema"] = df.versi_skema.astype("int64")
    df["ada_balasan"] = df.ada_balasan.astype("boolean")
    for k in ["dibuat_pada", "dibalas_pada"]:                      # +07:00 -> WIB naif, seperti waktu_wib Modul 4
        df[k] = (pd.to_datetime(df[k], utc=True).dt.tz_convert("Asia/Jakarta").dt.tz_localize(None)
                 .astype("datetime64[us]"))
    return df.sort_values("id_ulasan").reset_index(drop=True)


ulasan = bingkai_terpadu(kol.find())
assert len(ulasan) == kol.count_documents({}) and ulasan.id_ulasan.is_unique
assert ulasan.bintang.between(1, 5).all() and ulasan.bintang.notna().all()
cek = ulasan.groupby(["versi_skema", "bintang"]).size().reset_index(name="n")
assert np.array_equal(cek.to_numpy(dtype="int64"), per_bintang.to_numpy(dtype="int64"))     # pandas = pipeline
ulasan.to_parquet(OUT / "ulasan_terpadu_m08.parquet", index=False)
B.update(ulasan_tamu=int(ulasan.id_pelanggan.isna().sum()),
         rata_bintang=round(float(ulasan.bintang.mean()), 3))
print(ulasan.shape, "->", (OUT / "ulasan_terpadu_m08.parquet").name)
print("kosong per kolom:", ulasan.isna().sum()[lambda s: s > 0].to_dict())
ulasan.head(3)

**Pembahasan C**

- **Koleksi menerima semuanya.** 9.000 dokumen masuk, 0 ditolak: 3.892 berskema v1 dan 5.108
  berskema v2 (1.559 di antaranya memuat `balasan_penjual`), tiga bentuk dokumen dalam satu
  koleksi. Tidak ada yang memberi tahu bahwa bentuknya berubah; `skema_ulasan_m08.csv` adalah
  skema yang **ditemukan**, bukan yang dideklarasikan.
- **Kueri yang ditulis untuk satu versi diam-diam menjawab sebagian.** `{"rating": {"$lte": 2}}`
  mengembalikan 82 dokumen dan `{"penilaian.bintang": {"$lte": 2}}` 125; tidak ada galat, dan
  masing-masing tampak seperti jawaban lengkap. Ulasan berbintang ≤ 2 sebenarnya **207**. Pada
  penyimpanan dokumen, field yang tidak ada bukan kesalahan: ia hanya tidak cocok.
- **Kosong punya tiga wujud.** Tidak ada dokumen yang kehilangan field id pelanggan
  (`$exists: false` → 0), tetapi **144** dokumen memuat id berupa **string kosong** (ulasan
  pembeli tamu) dan 20 dokumen v2 memuat `produk.kategori: null` (produk yang tidak ada di
  katalog). `$exists` hanya menjawab "field-nya ada", bukan "nilainya ada".
- **`$unwind` mengubah butir.** Dari 5.108 dokumen v2, 1.735 tanpa tag **hilang** dan 1.679
  bertag dua **terhitung dua kali**: 5.052 baris. "Rata-rata bintang per tag" karenanya bukan
  rata-rata atas ulasan. Dengan `preserveNullAndEmptyArrays` kelompok "(tanpa tag)" muncul dan
  tidak ada yang hilang. Rata-rata bintang antartag nyaris sama (4,19–4,25): tag pada data ini
  tidak membedakan kepuasan.
- **Operator yang tidak didukung mongomock** dicatat di `mongomock_m08.csv` beserta cara lainnya:
  `$sortByCount`, `$dateFromString`, `$toDate`, `$round`, `$expr` pada jalur yang tidak ada,
  `$rename` ke jalur bertitik, dan `explain()`. `$lookup` didukung tetapi tanpa indeks. Kode yang
  sama berjalan di MongoDB sungguhan; yang berbeda hanya daftar ini.
- **Pandangan terpadu** `ulasan_terpadu_m08.parquet`: 9.000 baris × 17 kolom. Yang tidak direkam
  v1 (aspek, tag, balasan, kanal, kategori) menjadi kosong, **bukan** nol atau `False`: v1 tidak
  menyatakan "tanpa tag", ia tidak mencatat tag. `n_tag = 0` (v2 tanpa tag) dan `n_tag` kosong
  (v1) adalah dua hal berbeda. Hitungan per (versi, bintang) dari pandas sama dengan pipeline
  `$ifNull`: dua jalan menuju angka yang sama adalah uji yang murah.

## Bagian D — Integrasi lintas paradigma

### D.1 Ulasan masuk ke basis data relasional

In [ ]:
SKEMA_ULASAN = """
CREATE TABLE IF NOT EXISTS ulasan (
    id_ulasan           TEXT PRIMARY KEY,
    versi_skema         INTEGER NOT NULL CHECK (versi_skema IN (1, 2)),
    id_pesanan          TEXT NOT NULL,                                    -- seperti tertulis di dokumen
    id_pesanan_kanonik  TEXT NOT NULL REFERENCES pesanan (id_pesanan),
    kode_produk         TEXT NOT NULL REFERENCES produk (kode_produk),
    id_pelanggan        TEXT REFERENCES akun (id_pelanggan),
    bintang             INTEGER NOT NULL CHECK (bintang BETWEEN 1 AND 5),
    aspek_kualitas      INTEGER CHECK (aspek_kualitas IS NULL OR aspek_kualitas BETWEEN 1 AND 5),
    aspek_pengiriman    INTEGER CHECK (aspek_pengiriman IS NULL OR aspek_pengiriman BETWEEN 1 AND 5),
    teks                TEXT,
    n_media             INTEGER NOT NULL CHECK (n_media >= 0),
    dibuat_pada         TEXT NOT NULL
);
"""
KOLOM_ULASAN = ["id_ulasan", "versi_skema", "id_pesanan", "id_pesanan_kanonik", "kode_produk", "id_pelanggan",
                "bintang", "aspek_kualitas", "aspek_pengiriman", "teks", "n_media", "dibuat_pada"]


def muat_ulasan(kanonik: pd.Series) -> pd.DataFrame:
    """Muat ulang tabel ulasan dengan id_pesanan_kanonik yang diberikan; kembalikan tolakannya."""
    df = ulasan.assign(id_pesanan_kanonik=kanonik.to_numpy(),
                       dibuat_pada=ulasan.dibuat_pada.dt.strftime("%Y-%m-%d %H:%M:%S"))[KOLOM_ULASAN]
    with buka_db() as con:
        for v in ("v_rating", "v_ulasan_baris"):
            con.execute(f"DROP VIEW IF EXISTS {v}")
        con.executescript(SKEMA_ULASAN)
        con.execute("DELETE FROM ulasan")
        con.execute("DELETE FROM tolakan WHERE tabel = 'ulasan'")
        masuk, ditolak = muat(con, "ulasan", df)
        tolak = pd.read_sql_query("SELECT * FROM tolakan WHERE tabel = 'ulasan' ORDER BY kunci", con)
    print(f"ulasan: {len(df)} = {masuk} masuk + {ditolak} ditolak |", tolak.aturan.value_counts().to_dict())
    return tolak


tolak_ulasan = muat_ulasan(ulasan.id_pesanan)          # percobaan pertama: id pesanan apa adanya

ditolak_pesanan = pd.DataFrame([json.loads(x) for x in tolak_ulasan.isi[tolak_ulasan.aturan.str.contains("pesanan")]])
B.update(ulasan_tolak_awal=len(tolak_ulasan), ulasan_tolak_pesanan=len(ditolak_pesanan),
         pola_9=int(ditolak_pesanan.id_pesanan.str.contains(r"-9\d{5}$").sum()))
print("id pesanan yang ditolak:", ditolak_pesanan.id_pesanan.head(4).tolist(), f"| berpola 9xxxxx: {B['pola_9']}")

# Id berpola 9xxxxx adalah salinan dekat yang dibuang Modul 4; tabel duplikatnya memetakan
# id yang dibuang ke id yang dipertahankan.
PETA_DUP: dict = {}
if (M04 / "duplikat_transaksi_m04.csv").exists():
    dup = pd.read_csv(M04 / "duplikat_transaksi_m04.csv", dtype=str)
    dekat = dup[dup.jenis.eq("dekat")]
    PETA_DUP = dict(zip(dekat.id_pesanan_dibuang, dekat.id_pesanan_dipertahankan))
else:
    print("PERINGATAN: duplikat_transaksi_m04.csv tidak ada; ulasan ber-id pesanan duplikat tetap ditolak.")
kanonik = ulasan.id_pesanan.map(PETA_DUP).fillna(ulasan.id_pesanan)
tolak_ulasan = muat_ulasan(kanonik)                     # percobaan kedua: id dipetakan lewat bukti Modul 4
with buka_db() as con:
    tolakan = pd.read_sql_query("SELECT * FROM tolakan ORDER BY tabel, kunci", con)
    n_ulasan_db = con.execute("SELECT COUNT(*) FROM ulasan").fetchone()[0]
    kembar = pd.read_sql_query(
        "SELECT id_pesanan_kanonik, kode_produk, COUNT(*) AS n, COUNT(DISTINCT bintang) AS n_bintang, "
        "COUNT(DISTINCT teks) AS n_teks FROM ulasan GROUP BY 1, 2 HAVING COUNT(*) > 1", con)
tolakan.to_csv(OUT / "tolakan_m08.csv", index=False)
hasil_muat = pd.concat([hasil_muat[hasil_muat.tabel != "ulasan"],
                        pd.DataFrame([{"tabel": "ulasan", "baris_sumber": len(ulasan), "masuk": n_ulasan_db,
                                       "ditolak": len(tolak_ulasan)}])], ignore_index=True)
hasil_muat.to_csv(OUT / "muat_m08.csv", index=False)
B.update(ulasan_dipetakan=int((kanonik != ulasan.id_pesanan).sum()), ulasan_masuk=n_ulasan_db,
         ulasan_tolak=len(tolak_ulasan), kembar=len(kembar), kembar_beda_bintang=int((kembar.n_bintang > 1).sum()),
         kembar_beda_teks=int((kembar.n_teks > 1).sum()))
print(f"dipetakan lewat tabel duplikat Modul 4: {B['ulasan_dipetakan']} ulasan; "
      f"pasangan ulasan kembar (pesanan dan produk sama): {B['kembar']}, "
      f"bintangnya berbeda pada {B['kembar_beda_bintang']}, teksnya berbeda pada {B['kembar_beda_teks']}")

### D.2 Menggabungkan ulasan dengan baris pesanan

In [ ]:
Q_GABUNG = """
SELECT u.id_ulasan, b.baris_ke, u.bintang, b.rating_ulasan
FROM ulasan u
JOIN baris_pesanan b ON b.id_pesanan = u.id_pesanan_kanonik AND b.kode_produk = u.kode_produk
"""
gabung = kueri(Q_GABUNG)
with buka_db() as con:
    n_ulasan_db = con.execute("SELECT COUNT(*) FROM ulasan").fetchone()[0]
print(f"ulasan di basis data: {n_ulasan_db} | baris hasil JOIN: {len(gabung)} | id_ulasan unik: {gabung.id_ulasan.nunique()}")

In [ ]:
# Kunci gabung (id_pesanan, kode_produk) bukan kunci baris_pesanan: produk yang sama dapat muncul
# di dua baris satu pesanan. Aturan: pilih baris yang rating_ulasan-nya terisi, lalu baris_ke terkecil.
SKEMA_VIEW = """
DROP VIEW IF EXISTS v_ulasan_baris;
CREATE VIEW v_ulasan_baris AS
SELECT id_ulasan, id_pesanan_kanonik AS id_pesanan, baris_ke, kode_produk, bintang, rating_ulasan, n_kandidat
FROM (
    SELECT u.id_ulasan, u.id_pesanan_kanonik, u.kode_produk, u.bintang, b.baris_ke, b.rating_ulasan,
           COUNT(b.baris_ke) OVER (PARTITION BY u.id_ulasan) AS n_kandidat,
           ROW_NUMBER() OVER (PARTITION BY u.id_ulasan
                              ORDER BY b.rating_ulasan IS NULL, b.baris_ke) AS urut
    FROM ulasan u
    LEFT JOIN baris_pesanan b ON b.id_pesanan = u.id_pesanan_kanonik AND b.kode_produk = u.kode_produk
)
WHERE urut = 1;
"""
with buka_db() as con:
    con.executescript(SKEMA_VIEW)
taut = kueri("SELECT * FROM v_ulasan_baris ORDER BY id_ulasan")
assert len(taut) == n_ulasan_db and taut.id_ulasan.is_unique
B.update(gabung_baris=len(gabung), ulasan_ganda=int((taut.n_kandidat > 1).sum()),
         ulasan_tanpa_baris=int((taut.n_kandidat == 0).sum()),
         pasangan_ganda=int(transaksi.duplicated(["id_pesanan", "kode_produk"]).sum()))
print(f"{B['ulasan_ganda']} ulasan cocok dengan dua baris pesanan ({B['pasangan_ganda']} pasangan pesanan-produk"
      f" berulang di transaksi); {B['ulasan_tanpa_baris']} ulasan tanpa baris pesanan")
taut[taut.n_kandidat > 1]

### D.3 Konflik bintang dan sumber kebenaran

In [ ]:
Q_KONFLIK = """
SELECT v.id_ulasan, v.id_pesanan, v.baris_ke, v.kode_produk, u.versi_skema,
       v.bintang AS bintang_ulasan, v.rating_ulasan AS rating_transaksi,
       v.bintang - v.rating_ulasan AS selisih, u.aspek_kualitas, u.aspek_pengiriman
FROM v_ulasan_baris v
JOIN ulasan u ON u.id_ulasan = v.id_ulasan
WHERE v.rating_ulasan IS NOT NULL AND v.bintang <> v.rating_ulasan
ORDER BY v.id_ulasan
"""
konflik = kueri(Q_KONFLIK).astype({"aspek_kualitas": "Int64", "aspek_pengiriman": "Int64"})
sebanding = int(taut.rating_ulasan.notna().sum())

# Bukti: pada skema v2 pembeli juga menilai dua aspek. Sumber yang benar seharusnya dekat dengan aspeknya.
k2 = konflik[konflik.versi_skema.eq(2)]
jarak = lambda acuan: ((k2.aspek_kualitas - acuan).abs() + (k2.aspek_pengiriman - acuan).abs()) / 2   # noqa: E731
jauh = lambda acuan: int((((k2.aspek_kualitas - acuan).abs() > 1) | ((k2.aspek_pengiriman - acuan).abs() > 1)).sum())   # noqa: E731
B.update(konflik=len(konflik), sebanding=sebanding, konflik_v1=int(konflik.versi_skema.eq(1).sum()),
         konflik_v2=len(k2), jarak_ulasan=round(float(jarak(k2.bintang_ulasan).mean()), 2),
         jarak_transaksi=round(float(jarak(k2.rating_transaksi).mean()), 2),
         jauh_ulasan=jauh(k2.bintang_ulasan), jauh_transaksi=jauh(k2.rating_transaksi),
         selisih=konflik.selisih.value_counts().sort_index().to_dict())
konflik["sumber_dipilih"] = "ulasan"
konflik["nilai_dipilih"] = konflik.bintang_ulasan
konflik.to_csv(OUT / "konflik_rating_m08.csv", index=False)
print(f"konflik: {B['konflik']} dari {sebanding} ulasan yang dapat dibandingkan "
      f"({B['konflik'] / sebanding:.1%}); v1 {B['konflik_v1']}, v2 {B['konflik_v2']}; selisih {B['selisih']}")
print(f"v2: rata-rata jarak aspek ke bintang ulasan {B['jarak_ulasan']}, ke rating transaksi {B['jarak_transaksi']};"
      f" aspek berjarak > 1: {B['jauh_ulasan']} vs {B['jauh_transaksi']} dari {B['konflik_v2']}")

silang = pd.crosstab(taut.rating_ulasan[taut.rating_ulasan.notna()].astype(int), taut.bintang).reindex(
    index=range(1, 6), columns=range(1, 6), fill_value=0)
fig, ax = plt.subplots(figsize=(5.2, 4.3))
ax.imshow(np.log10(silang.to_numpy() + 1), cmap="Blues")
for i in range(5):
    for j in range(5):
        n = int(silang.iat[i, j])
        if n:
            ax.text(j, i, f"{n:,}".replace(",", "."), ha="center", va="center", fontsize=9,
                    color="white" if n > 500 else "#b23a48" if i != j else "black",
                    fontweight="bold" if i != j else "normal")
ax.set_xticks(range(5), range(1, 6))
ax.set_yticks(range(5), range(1, 6))
ax.set_xlabel("bintang pada dokumen ulasan")
ax.set_ylabel("rating_ulasan pada baris pesanan")
ax.set_title("Dua catatan untuk penilaian yang sama")
fig.tight_layout()
fig.savefig(OUT / "03_konflik_rating.png", dpi=110)
plt.show()

# Keputusan diwujudkan sebagai view: nilai teramati pada baris_pesanan tidak diubah.
SKEMA_RATING = """
DROP VIEW IF EXISTS v_rating;
CREATE VIEW v_rating AS
SELECT b.id_pesanan, b.baris_ke, b.kode_produk, b.rating_ulasan AS rating_transaksi,
       x.bintang AS bintang_ulasan, x.id_ulasan,
       COALESCE(x.bintang, b.rating_ulasan) AS rating,
       CASE WHEN x.bintang IS NOT NULL THEN 'ulasan'
            WHEN b.rating_ulasan IS NOT NULL THEN 'transaksi' END AS sumber_rating
FROM baris_pesanan b
LEFT JOIN (
    SELECT v.id_pesanan, v.baris_ke, v.id_ulasan, v.bintang,
           ROW_NUMBER() OVER (PARTITION BY v.id_pesanan, v.baris_ke
                              ORDER BY u.id_pesanan <> u.id_pesanan_kanonik, v.id_ulasan) AS urut
    FROM v_ulasan_baris v JOIN ulasan u ON u.id_ulasan = v.id_ulasan
    WHERE v.baris_ke IS NOT NULL
) x ON x.id_pesanan = b.id_pesanan AND x.baris_ke = b.baris_ke AND x.urut = 1;
"""
with buka_db() as con:
    con.executescript(SKEMA_RATING)
    rating = pd.read_sql_query("SELECT sumber_rating, COUNT(*) AS n FROM v_rating GROUP BY 1", con)
    n_v, n_b = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("v_rating", "baris_pesanan")]
    yatim_db = con.execute("PRAGMA foreign_key_check").fetchall()
assert n_v == n_b and not yatim_db
B["sumber_rating"] = {(k if isinstance(k, str) else "tanpa rating"): int(v) for k, v in zip(rating.sumber_rating, rating.n)}
print("v_rating:", B["sumber_rating"])

# Pandangan terpadu dilengkapi kolom tautan; tetap satu baris per dokumen.
ditolak_produk = set(tolak_ulasan.kunci[tolak_ulasan.aturan.str.contains("produk")])
ditolak_lain = set(tolak_ulasan.kunci) - ditolak_produk
t = taut.set_index("id_ulasan")
ulasan["id_pesanan_kanonik"] = kanonik.to_numpy()
ulasan["baris_ke"] = ulasan.id_ulasan.map(t.baris_ke).astype("Int64")
ulasan["rating_transaksi"] = ulasan.id_ulasan.map(t.rating_ulasan).astype("Int64")
ulasan["status_tautan"] = np.select(
    [ulasan.id_ulasan.isin(ditolak_produk), ulasan.id_ulasan.isin(ditolak_lain),
     ulasan.id_pesanan_kanonik != ulasan.id_pesanan, ulasan.id_ulasan.map(t.n_kandidat).gt(1)],
    ["produk_tidak_dikenal", "pesanan_tidak_dikenal", "pesanan_duplikat_m04", "baris_ganda"], "cocok")
ulasan["status_tautan"] = ulasan.status_tautan.astype("str")
ulasan["konflik"] = ulasan.id_ulasan.isin(konflik.id_ulasan)
assert len(ulasan) == len(DOK)
ulasan.to_parquet(OUT / "ulasan_terpadu_m08.parquet", index=False)
B["status_tautan"] = ulasan.status_tautan.value_counts().to_dict()
print("status_tautan:", B["status_tautan"])

### D.4 SQL atau dokumen? Lima kebutuhan Lapak Sumatera

In [ ]:
perbandingan = pd.DataFrame([
    {"kebutuhan": "laporan omzet per provinsi per bulan", "pilihan": "sql",
     "alasan": "agregasi lintas tabel dengan butir yang dijamin kunci primer; hasilnya dapat direkonsiliasi",
     "bukti": f"JOIN 3 tabel = pandas pada {B['baris_masuk']} baris; selisih {rp(B['selisih_omzet'])} terjelaskan oleh {B['tolak_selesai']} baris tolakan"},
    {"kebutuhan": "menolak baris pesanan yang produknya tidak ada di katalog", "pilihan": "sql",
     "alasan": "kunci asing menagih aturan saat menulis; koleksi dokumen menerima apa pun",
     "bukti": f"{B['tolak_fk']} baris ditolak FK, 1 ditolak NOT NULL; mongomock menerima {B['n_dok']} dokumen, 0 ditolak, termasuk {B['id_kosong']} id pelanggan kosong"},
    {"kebutuhan": "menyimpan ulasan yang bentuknya berubah (media, tag, balasan penjual)", "pilihan": "dokumen",
     "alasan": "bentuk baru masuk tanpa migrasi; array dan objek bersarang disimpan apa adanya",
     "bukti": f"{B['n_bentuk']} bentuk dokumen dalam satu koleksi (v1 {B['n_v1']}, v2 {B['n_v2']}, {B['n_balasan']} berbalasan); harga: kueri satu versi hanya menemukan {B['rendah_v1']} atau {B['rendah_v2']} dari {B['rendah']} ulasan bintang <= 2"},
    {"kebutuhan": "menampilkan satu pesanan lengkap (kepala, baris, ulasan) dalam satu baca", "pilihan": "dokumen",
     "alasan": "data yang selalu dibaca bersama disimpan bersama; di SQL perlu JOIN beberapa tabel",
     "bukti": f"{B['pesanan']} pesanan tersebar di pesanan, baris_pesanan ({B['baris_masuk']} baris), dan ulasan ({B['ulasan_masuk']} baris)"},
    {"kebutuhan": "mengoreksi nama toko atau kategori produk di satu tempat", "pilihan": "sql",
     "alasan": "3NF menyimpan setiap fakta sekali; salinan di dalam dokumen harus diperbarui satu per satu",
     "bukti": f"kategori tersalin pada {B['n_v2']} dokumen ulasan v2 ({B['kategori_null']} di antaranya null); di SQL cukup 1 baris tabel produk"},
])
perbandingan.to_csv(OUT / "perbandingan_m08.csv", index=False)
perbandingan

### D.5 Log keputusan

In [ ]:
KOLOM_LOG = ["id", "modul", "sumber_kolom", "temuan", "bukti", "tindakan", "alasan",
             "baris_terdampak", "id_anomali"]


def catat_keputusan(baris: list[dict], path: Path = LOG) -> pd.DataFrame:
    """Tulis ulang baris log dengan id yang sama agar notebook aman dijalankan berulang."""
    lama_log = (pd.read_csv(path, dtype=str, keep_default_na=False) if path.exists()
                else pd.DataFrame(columns=KOLOM_LOG))
    baru = pd.DataFrame(baris, columns=KOLOM_LOG).astype(str)
    log = pd.concat([lama_log[~lama_log.id.isin(baru.id)], baru]).sort_values("id")
    log.to_csv(path, index=False)
    if path.parent != MODUL:
        shutil.copyfile(path, MODUL / path.name)   # salinan untuk ZIP tugas
    return log

In [ ]:
log = catat_keputusan([
    {"id": "K-08-001", "modul": 8, "sumber_kolom": "sqlite.toko;sqlite.produk;produk_m04.provinsi_toko;produk_m04.rating_toko",
     "temuan": "provinsi_toko dan rating_toko tidak bergantung pada id_toko",
     "bukti": f"{B['toko_multi_provinsi']} dari {B['n_toko_katalog']} toko tercatat di lebih dari satu provinsi; {B['toko_multi_rating']} toko dengan lebih dari satu rating",
     "tindakan": "disimpan di tabel produk sebagai provinsi_asal dan rating_katalog; profil toko diambil dari toko_m07" if ADA_M07
                 else "disimpan di tabel produk sebagai provinsi_asal dan rating_katalog; kolom profil toko dibiarkan NULL",
     "alasan": "menaruhnya di tabel toko berarti memilih satu nilai dan membuang sisanya tanpa catatan; modus ditolak karena menyembunyikan ketidakkonsistenan",
     "baris_terdampak": 0, "id_anomali": "-"},
    {"id": "K-08-002", "modul": 8, "sumber_kolom": "sqlite.pelanggan;sqlite.akun;pelanggan_m04.id_entitas",
     "temuan": "nama, email, dan tanggal daftar berbeda antar-id dalam satu entitas",
     "bukti": f"entitas dengan lebih dari satu nilai: nama {B['entitas_multi_nama']}, email {B['entitas_multi_email']}, tanggal daftar {B['entitas_multi_tanggal']}; no_hp, kota, jenis kelamin, tahun lahir: 0",
     "tindakan": f"dua tabel: pelanggan ({len(T['pelanggan'])} entitas) dan akun ({len(T['akun'])} id sumber); pesanan merujuk akun",
     "alasan": "atribut yang berbeda antar-id adalah fakta tentang akun; satu tabel per entitas akan membuang nama dan email akun kedua",
     "baris_terdampak": 0, "id_anomali": "-"},
    {"id": "K-08-003", "modul": 8, "sumber_kolom": "sqlite.pesanan;transaksi_m04.waktu_wib;transaksi_m04.lama_kirim_hari",
     "temuan": "atribut pesanan direkam berulang di setiap baris dan kosong pada sebagian barisnya",
     "bukti": f"tidak pernah bernilai dua dalam satu pesanan; kosong sebagian pada {B['pesanan_waktu_sebagian']} pesanan (waktu_wib) dan {B['pesanan_lama_sebagian']} pesanan (lama_kirim_hari)",
     "tindakan": "disimpan sekali di tabel pesanan: waktu terkecil dan lama kirim yang terisi; ongkir dan rating_ulasan tetap per baris",
     "alasan": f"ongkir berbeda antarbaris pada {B['pesanan_multi_ongkir']} pesanan; rating kosong sebagian pada {B['pesanan_rating_sebagian']} pesanan berarti barang itu tidak dinilai, bukan gagal tercatat, jadi keduanya atribut baris",
     "baris_terdampak": 0, "id_anomali": "-"},
    {"id": "K-08-004", "modul": 8, "sumber_kolom": "sqlite.baris_pesanan.kode_produk;sqlite.baris_pesanan.jumlah;sqlite.tolakan",
     "temuan": "baris pesanan melanggar kunci asing produk atau NOT NULL jumlah",
     "bukti": f"{B['baris_sumber']} = {B['baris_masuk']} masuk + {B['baris_tolak']} ditolak ({B['tolak_fk']} FK kode_produk, {B['tolak_null']} NOT NULL jumlah); nilai tolakan {rp(B['total_tolak'])}; tanpa PRAGMA foreign_keys {B['yatim_tanpa_pragma']} baris yatim masuk diam-diam",
     "tindakan": "ditolak constraint dan dicatat pada tabel tolakan beserta pesan basis data; PRAGMA foreign_keys = ON pada setiap koneksi",
     "alasan": "produk yatim tidak ditambahkan ke katalog tanpa bukti; baris tidak dihapus diam-diam karena nilainya nyata",
     "baris_terdampak": B["baris_tolak"], "id_anomali": "A-16"},
    {"id": "K-08-005", "modul": 8, "sumber_kolom": "sqlite.pesanan;sqlite.baris_pesanan",
     "temuan": "pesanan tanpa baris sesudah penolakan, dan omzet SQL lebih kecil dari pandas",
     "bukti": f"{B['pesanan_kosong']} dari {B['pesanan']} pesanan tidak memiliki baris; omzet selesai SQL lebih kecil {rp(B['selisih_omzet'])} = {B['tolak_selesai']} baris selesai yang ditolak",
     "tindakan": "kepala pesanan dipertahankan; selisih dilaporkan bersama tabel tolakan pada setiap laporan omzet",
     "alasan": "kepala pesanan sah; menghapusnya akan menghilangkan jejak bahwa pesanan itu ada",
     "baris_terdampak": 0, "id_anomali": "A-16"},
    {"id": "K-08-006", "modul": 8, "sumber_kolom": "sqlite.baris_pesanan;sqlite.pesanan (indeks)",
     "temuan": "kolom kunci asing tidak otomatis berindeks; fungsi pada kolom mematikan indeks",
     "bukti": f"rencana awal SCAN untuk kode_produk dan id_pelanggan; sesudah indeks SEARCH; substr(waktu_pesan) tetap SCAN, bentuk rentang SEARCH ({B['n_bulan']} pesanan sama); berkas {B['ukuran_tanpa_indeks'] // 1024} -> {B['ukuran_dengan_indeks'] // 1024} KiB",
     "tindakan": "tiga indeks dibuat (kode_produk, id_pelanggan, waktu_pesan); kueri bulan ditulis sebagai rentang",
     "alasan": "indeks hanya untuk kueri yang rencananya terbukti berubah; setiap indeks menambah ukuran dan biaya tulis",
     "baris_terdampak": 0, "id_anomali": "-"},
    {"id": "K-08-007", "modul": 8, "sumber_kolom": "mongo.ulasan.rating;mongo.ulasan.penilaian.bintang;mongo.ulasan.tag",
     "temuan": "dua versi skema dalam satu koleksi; kueri satu versi menjawab sebagian tanpa galat",
     "bukti": f"v1 {B['n_v1']}, v2 {B['n_v2']}; bintang <= 2: {B['rendah_v1']} (v1) + {B['rendah_v2']} (v2) = {B['rendah']}; $unwind tag membuang {B['tanpa_tag']} dokumen dan menghitung {B['dua_tag']} dokumen dua kali",
     "tindakan": "pandangan terpadu ulasan_terpadu_m08.parquet lewat terpadu(); versi yang belum dikenal menghentikan notebook",
     "alasan": "schema-on-read memindahkan tanggung jawab skema ke setiap pembaca; satu fungsi pemetaan lebih aman daripada $or di setiap kueri",
     "baris_terdampak": 0, "id_anomali": "-"},
    {"id": "K-08-008", "modul": 8, "sumber_kolom": "mongo.ulasan.id_pelanggan;mongo.ulasan.pelanggan.id;mongo.ulasan.produk.kategori",
     "temuan": "id pelanggan berupa string kosong dan kategori null pada dokumen",
     "bukti": f"{B['id_kosong']} dokumen ber-id pelanggan string kosong ($exists false: {B['id_hilang']}); {B['kategori_null']} dokumen v2 berkategori null",
     "tindakan": "string kosong dibaca sebagai kosong (NULL) pada pandangan terpadu; field yang tidak direkam v1 dibiarkan kosong, bukan 0 atau False",
     "alasan": "string kosong akan melanggar kunci asing ke akun; mengisi 0 untuk tag v1 mengarang fakta yang tidak pernah dicatat",
     "baris_terdampak": B["id_kosong"], "id_anomali": "-"},
    {"id": "K-08-009", "modul": 8, "sumber_kolom": "sqlite.ulasan.id_pesanan_kanonik;mongo.ulasan;duplikat_transaksi_m04",
     "temuan": "ulasan merujuk pesanan atau produk yang tidak ada di basis data",
     "bukti": f"muat pertama menolak {B['ulasan_tolak_awal']} ulasan: {B['ulasan_tolak_pesanan']} id pesanan berpola 9xxxxx (salinan dekat yang dibuang Modul 4) dan {B['ulasan_tolak']} kode produk yatim; sesudah dipetakan {B['ulasan_masuk']} masuk",
     "tindakan": f"{B['ulasan_dipetakan']} id pesanan dipetakan ke id yang dipertahankan lewat duplikat_transaksi_m04.csv (id asli tetap disimpan); {B['ulasan_tolak']} ulasan produk yatim ditolak dan dicatat; {B['kembar']} pasang ulasan kembar dipertahankan keduanya",
     "alasan": "pemetaan memiliki bukti tertulis dari Modul 4; ulasan kembar tidak dibuang karena tidak ada bukti mana yang asli",
     "baris_terdampak": B["ulasan_dipetakan"], "id_anomali": "A-16"},
    {"id": "K-08-010", "modul": 8, "sumber_kolom": "sqlite.v_rating;sqlite.baris_pesanan.rating_ulasan;mongo.ulasan.penilaian.bintang",
     "temuan": "bintang pada ulasan berbeda dari rating_ulasan pada baris pesanan",
     "bukti": f"{B['konflik']} konflik dari {B['sebanding']} ulasan yang dapat dibandingkan (v1 {B['konflik_v1']}, v2 {B['konflik_v2']}); pada v2 jarak rata-rata aspek ke bintang ulasan {angka(B['jarak_ulasan'], 2)} dan ke rating transaksi {angka(B['jarak_transaksi'], 2)}; {B['ulasan_ganda']} ulasan cocok dengan dua baris",
     "tindakan": "sumber kebenaran bintang = dokumen ulasan, diwujudkan sebagai view v_rating; baris_pesanan.rating_ulasan tidak diubah",
     "alasan": "aspek yang ditulis pembeli konsisten dengan bintang ulasan, tidak dengan rating transaksi; menimpa kolom transaksi menghapus bukti konflik",
     "baris_terdampak": 0, "id_anomali": "-"},
])
log[log.id.str.startswith("K-08")][["id", "sumber_kolom", "baris_terdampak"]]

**Pembahasan D**

- **Kunci asing menagih lagi di perbatasan dua penyimpanan.** Muat pertama menolak **65** ulasan:
  38 merujuk id pesanan yang tidak ada, 27 merujuk kode produk yatim. Ke-38 id itu berpola
  `LS-tttt-9xxxxx`, yaitu salinan dekat yang dibuang Modul 4. `duplikat_transaksi_m04.csv`
  memetakan setiap id yang dibuang ke id yang dipertahankan, sehingga ada **bukti** untuk
  memetakannya: `id_pesanan_kanonik` menunjuk pesanan yang dipertahankan dan `id_pesanan` asli
  tetap disimpan. Muat kedua: 9.000 = 8.973 masuk + 27 ditolak. Keputusan membuang duplikat di
  Modul 4 ternyata memutus rujukan di penyimpanan lain; tanpa tabel duplikat, 38 ulasan itu hilang.
- **Pemetaan itu memunculkan ulasan kembar.** 22 pasang ulasan kini merujuk pesanan dan produk
  yang sama (satu dari pesanan asli, satu dari salinannya); pada 18 pasang teksnya berbeda dan
  pada 1 pasang bintangnya berbeda. Tidak ada bukti mana yang asli, jadi keduanya dipertahankan.
- **JOIN dapat menggandakan baris.** `(id_pesanan, kode_produk)` bukan kunci `baris_pesanan`:
  produk yang sama muncul di dua baris pada 10 pasangan pesanan–produk, dan **4** ulasan cocok
  dengan dua baris (8.973 ulasan → 8.977 baris hasil JOIN). `v_ulasan_baris` memilih baris yang
  `rating_ulasan`-nya terisi, lalu `baris_ke` terkecil, dan membuktikan satu baris per ulasan.
- **Konflik bintang: 256 dari 8.973 ulasan yang dapat dibandingkan (2,9%)**, dengan selisih −2
  (151), +1 (76), +2 (28), dan −1 (1); kedua versi skema terkena kira-kira sama (v1 105, v2 151),
  jadi ini bukan cacat migrasi skema.
- **Sumber kebenaran dipilih dengan bukti, bukan selera.** Pada 151 konflik berskema v2, pembeli
  juga menilai dua aspek. Jarak rata-rata aspek ke bintang ulasan 0,51, ke `rating_ulasan`
  transaksi 1,52; aspek berjarak lebih dari 1 dari bintang ulasan pada 0 konflik, dari rating
  transaksi pada 94 konflik. Dokumen ulasan konsisten dengan dirinya sendiri; kolom transaksi
  adalah salinan yang tertinggal. **Bintang dipercaya dari ulasan.** Untuk v1 tidak ada aspek;
  keputusan yang sama diterapkan dengan catatan bahwa buktinya tidak langsung.
- **Keputusan diwujudkan sebagai `VIEW v_rating`**, bukan `UPDATE`: `baris_pesanan.rating_ulasan`
  tetap nilai teramati, dan konfliknya tetap dapat diaudit di `konflik_rating_m08.csv`. Ulasan
  hanya mencakup 8.951 baris pesanan; 4.829 baris berating tanpa ulasan memakai rating transaksi,
  10.143 baris tidak berating, dan kolom `sumber_rating` menyebut asalnya.
- **Tidak ada pemenang tunggal** pada tabel lima kebutuhan: agregasi, penolakan, dan koreksi satu
  tempat memihak SQL; bentuk yang berubah dan "satu pesanan dalam satu baca" memihak dokumen.
  Harga memakai keduanya adalah menjaga agar keduanya tetap sepakat, dan itulah pekerjaan Tugas.

### D.6 Isi basis data sesudah Bagian D

In [ ]:
with buka_db() as con:
    objek = con.execute("SELECT name, type FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' "
                        "ORDER BY type, name").fetchall()
    isi_db = pd.DataFrame([{"objek": nama, "jenis": jenis,
                            "baris": con.execute(f"SELECT COUNT(*) FROM {nama}").fetchone()[0]
                            if jenis in ("table", "view") else None} for nama, jenis in objek])
    print("foreign_key_check:", len(con.execute("PRAGMA foreign_key_check").fetchall()), "pelanggaran |",
          "integrity_check:", con.execute("PRAGMA integrity_check").fetchone()[0])
isi_db.astype({"baris": "Int64"})

## Latihan 8.1 — Pesanan sebagai dokumen: *embed* atau *reference*

**Dinilai.** Kerjakan di laboratorium; jawabannya ditulis pada **`dw-module-08/latihan-dokumen.md`**
dan diunggah ke tugas *Latihan Mandiri di Lab — Modul 08*.

1. Bangun dari `lapak_m08.sqlite` dua model dokumen pesanan di mongomock: **embed** (satu dokumen
   per pesanan, baris pesanan di dalam array `baris`) dan **reference** (koleksi kepala pesanan dan
   koleksi baris pesanan yang merujuk `id_pesanan`).
2. Hitung "omzet per provinsi tujuan, status selesai" di SQL, di model embed (pipeline agregasi),
   dan di model reference. Buktikan ketiganya sama.
3. Ukur waktu ketiganya dan ukuran kedua model (jumlah dokumen, ukuran JSON). Simpan
   `keluaran/model_dokumen_m08.csv` (`model`, `n_dokumen`, `ukuran_kb`, `waktu_ms`, `omzet_total`,
   `sama_dengan_sql`).
4. Putuskan untuk setiap bagian pesanan (baris, produk, pelanggan, ulasan): di-*embed* atau
   dirujuk, dengan alasan dan satu angka pendukung.

> **Peringatan** — `$lookup` mongomock tidak memakai indeks. Jangan menjalankannya pada seluruh
> koleksi baris pesanan; ukur pada cuplikan kecil, atau gabungkan di sisi aplikasi.

In [ ]:
def catatan(df: pd.DataFrame) -> list[dict]:
    """DataFrame -> daftar dict bertipe Python asli (NaN menjadi None), siap menjadi dokumen."""
    return [dict(zip(df.columns, r)) for r in rekaman(df)]


def ukur_ms(fungsi, ulang: int = 3) -> tuple[float, object]:
    """(median waktu dalam milidetik, hasil fungsi) dari `ulang` kali jalan."""
    lama = []
    for _ in range(ulang):
        t0 = time.perf_counter()
        hasil = fungsi()
        lama.append((time.perf_counter() - t0) * 1000)
    return float(np.median(lama)), hasil


kepala = kueri("SELECT p.id_pesanan, p.kanal, p.id_pelanggan, p.waktu_pesan, p.metode_bayar, p.kota_tujuan, "
               "w.provinsi AS provinsi_tujuan, p.kurir, p.status, p.lama_kirim_hari "
               "FROM pesanan p JOIN wilayah w ON w.kota = p.kota_tujuan ORDER BY p.id_pesanan"
               ).astype({"lama_kirim_hari": "Int64"})
baris_sql = kueri("SELECT b.*, pr.kategori FROM baris_pesanan b JOIN produk pr ON pr.kode_produk = b.kode_produk "
                  "ORDER BY b.id_pesanan, b.baris_ke").astype({"rating_ulasan": "Int64"})
# TODO: bangun dok_embed dan kedua koleksi reference, ukur ketiga cara, simpan model_dokumen_m08.csv

## Latihan 8.2 — Migrasi skema ulasan v1 → v2 dengan `update_many`

**Dinilai.** Jawabannya ditulis pada **`dw-module-08/latihan-migrasi.md`** dan diunggah ke tugas *Latihan Mandiri di Lab — Modul 08*.

1. Salin seluruh dokumen ke koleksi baru `ulasan_migrasi` (koleksi `ulasan` Bagian C tidak boleh
   berubah). Ubah setiap dokumen v1 menjadi bentuk v2 dengan `update_many`: `pesanan.id`,
   `produk.kode`, `pelanggan.id`, `penilaian.bintang`, dan `media` (array objek), lalu hapus field
   lama. Tandai dokumen hasil migrasi (`migrasi.dari_versi`).
2. Lengkapi `pesanan.kanal` dan `produk.kategori` dokumen hasil migrasi dari basis data SQLite.
   Field yang tidak pernah direkam v1 (aspek, tag) **tidak** boleh dikarang.
3. Validasi: tidak ada lagi field v1; banyak dokumen dan sebaran bintang tidak berubah;
   `bingkai_terpadu()` atas koleksi hasil migrasi sama dengan pandangan terpadu Bagian C untuk kolom
   yang ada di kedua versi. Kolom mana yang **berubah makna** sesudah migrasi?
4. Jalankan migrasi dua kali dan buktikan jalan kedua tidak mengubah apa pun. Simpan
   `keluaran/migrasi_m08.csv` (`langkah`, `cocok`, `diubah`, `diubah_jalan_kedua`).

> **Peringatan** — di mongomock, `update_one` di dalam perulangan ribuan kali dan `update_many`
> pada koleksi berindeks unik sangat lambat. Pakai beberapa `update_many` dengan `$in`, dan
> buat indeks (bila perlu) sesudah migrasi.

In [ ]:
mig = klien["lapak"]["ulasan_migrasi"]
mig.delete_many({})
mig.insert_many([dict(d) for d in DOK])
# TODO: migrasi v1 -> v2 dengan update_many, pelengkapan kanal dan kategori dari SQLite,
#       validasi, uji jalan kedua, dan keluaran/migrasi_m08.csv

## Tugas Individu — dua penyimpanan yang harus tetap sepakat

1. **Koleksi dokumen pesanan.** Dari `lapak_m08.sqlite`, bangun satu dokumen per pesanan: kepala
   pesanan, array `baris` (setiap baris memuat `baris_ke`, `produk.kode`, `jumlah`, `total_bayar`,
   dan, bila ada, `rating` menurut `v_rating` beserta `sumber_rating`), serta `ulasan`
   (`id_ulasan`, `bintang`) yang tertaut pada baris itu. Dokumen dibuat ramping: atribut produk
   dan pelanggan dirujuk dengan kodenya, tidak disalin. Simpan sebagai `keluaran/pesanan_dok_m08.jsonl` (satu dokumen per baris berkas, kunci
   `id_pesanan`).
2. **Uji konsistensi.** Muat berkas itu kembali ke mongomock, lalu bandingkan sekurang-kurangnya
   enam ukuran antara SQLite dan koleksi dokumen (banyak pesanan, banyak baris, unit, pesanan
   selesai, omzet selesai, ulasan tertaut, jumlah bintang). Simpan `keluaran/konsistensi_m08.csv`
   (`tahap`, `ukuran`, `nilai_sql`, `nilai_dokumen`, `selisih`) dengan `tahap = awal`.
3. **Simulasi perubahan.** Di dalam satu transaksi yang kemudian di-*rollback*, ubah status satu
   pesanan di SQLite, hitung ulang ukuran SQL, dan tambahkan hasilnya dengan
   `tahap = setelah_ubah_sql`. Ukuran mana yang kini berselisih? Buktikan basis data kembali
   seperti semula sesudah *rollback*.
4. Selesaikan Latihan 8.1 dan 8.2, lengkapi log keputusan, dan tulis `laporan.md`.

In [ ]:
# TODO: pesanan_dok_m08.jsonl, konsistensi_m08.csv (tahap awal dan setelah_ubah_sql), dan baris log
#       K-08 milik Anda sendiri untuk Latihan 8.1, Latihan 8.2, dan Tugas.